In [1]:
import json
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

In [2]:
data = pd.read_csv("data/all_bundestag_speeches_preprocessed.csv", delimiter=";", low_memory=False)

## Constructing an anti-migration lexicon

In [3]:
df_afd = data[data["Party"] == "AfD"].copy()

In [4]:
print(df_afd.shape)
print(df_afd["date"].min(), df_afd["date"].max())

(6261, 20)
2017-10-24 2025-06-27


In [5]:
df_afd["date"] = pd.to_datetime(df_afd["date"], errors="coerce")

In [6]:
df_afd["date_quarter"] = df_afd["date"].dt.to_period("Q")

In [3]:
data["date"] = pd.to_datetime(data["date"])

### Examining migration speech data from 2017

In [8]:

df_afd_2017 = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2017)
].copy()

In [9]:
df_afd_2017 

,speech_identification_ent,date,period,session,pos_speechbeginning,Party,Role,governing_Party,text,text_length,text_preprocessed,text_preprocessed_lemmatized,text_length_preprocessed,text_length_lemmatized,date_year,date_quarter,similarity_expansive,similarity_restrictive,periode_index,modified_flag
210382,Bernd Baumann,2017-10-24,19,1,NaN,AfD,MdB,0.0,Herr Präsident! Meine Damen und Herren! Immer ...,524,deutlicher zeigte verlauf afd einziehen alters...,dame verlauf afd einziehen alterspraesidenten ...,152,151,2017,4,-0.114863,-0.099890,2017Q4,False
210389,Bernd Baumann,2017-11-21,19,2,NaN,AfD,MdB,0.0,Herr Präsident! Meine Damen und Herren! Wir ha...,124,ausser angeschlossen unsicheren zeiten geschei...,dame ausser linke anschliessen unsicher gesche...,27,31,2017,4,-0.083938,-0.061264,2017Q4,False
210395,Jan Ralf Nolte,2017-11-21,19,2,NaN,AfD,MdB,0.0,Sehr geehrter Herr Präsident! Sehr geehrte Kol...,609,soldat tapfer verteidigen legt massstab moment...,soldat tapfer verteidigen legt massstab moment...,182,176,2017,4,-0.031857,-0.063415,2017Q4,False
210404,Norbert Kleinwächter,2017-11-21,19,2,NaN,AfD,MdB,0.0,Herr Präsident! Werte Kollegen! Dieser Einsatz...,617,werte einsatz markiert tiefpunkt diplomatie me...,einsatz markieren tiefpunkt diplomatie merkel ...,184,179,2017,4,0.042278,0.049116,2017Q4,False
210411,Eberhardt Alexander Gauland,2017-11-21,19,2,NaN,AfD,MdB,0.0,Herr Präsident! Meine Damen und Herren!\n\nMit...,497,beantragte mandat entsendung soldaten afghanis...,dame mandat entsendung soldat afghanistan daue...,147,150,2017,4,-0.002018,0.007914,2017Q4,False
210418,Gerold Otten,2017-11-21,19,2,NaN,AfD,MdB,0.0,Herr Präsident! Werte Kolleginnen und Kollegen...,688,werte region darfur schauplatz blutigen konfli...,region darfur schauplatz blutig konflikt mensc...,211,205,2017,4,-0.063011,-0.097076,2017Q4,False
210426,Gerold Otten,2017-11-21,19,2,NaN,AfD,MdB,0.0,Herr Präsident! Meine Damen und Herren! Seit e...,605,nachrichten erschreckende bilder ueberfuellten...,dame nachricht erschreckend bild ueberfuellt s...,174,180,2017,4,-0.106042,-0.128976,2017Q4,False
210434,Tino Chrupalla,2017-11-21,19,2,NaN,AfD,MdB,0.0,"Frau Präsidentin, zu meinem Namen. Er wird so ...",563,tino krupalla schlesischer name richtigkeit ha...,name tino krupalla schlesisch name richtigkeit...,170,166,2017,4,-0.102912,-0.072131,2017Q4,False
210440,Heiko Heßenkemper,2017-11-21,19,2,NaN,AfD,MdB,0.0,Frau Präsidentin! Meine Damen und Herren! In d...,439,beantragten angekuendigten werkschliessungen s...,dame aktuelle stunde angekuendigt werkschliess...,126,121,2017,4,-0.126388,-0.127065,2017Q4,False
210446,Harald Weyel,2017-11-21,19,2,NaN,AfD,MdB,0.0,Sehr geehrte Präsidentin! Sehr geehrte Damen u...,620,ruehrend minute afd streng rueckzahlung aeusse...,dame abgeordnete ruehren last minute afd stren...,149,164,2017,4,-0.029300,0.022561,2017Q4,False


In [10]:
migration_terms = [
    "fluechtling", "asyl", "migration", "einwanderung","islamist","Ausländer",
    "zuwanderung", "grenze", "abschiebung","abschieben","reimigration","Migranten",
    "integration", "aufenthalt", "asylantrag", "identitaet","Asylklagen","grenze","auswanderung",
    "asylbewerber","flucht","schutzsuchend","aufenthaltsrecht","aufenthaltstitel","duldung",
    "visa", "visum",
    "familiennachzug"
]

In [11]:
def contains_migration(text):
    text = str(text).lower()
    return any(term in text for term in migration_terms)

df_afd_2017_migration = df_afd_2017[
    df_afd_2017["text_preprocessed_lemmatized"].apply(contains_migration)
]

In [12]:
df_afd_2017_migration["text_preprocessed_lemmatized"].tolist()[221]

IndexError: list index out of range

In [ ]:
df_afd_2017_migration["text"].tolist()[224]

### Examining migration speech data from 2018

In [ ]:

df_afd_2018 = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2018)
].copy()


df_afd_2018_migration = df_afd_2018[
    df_afd_2018["text_preprocessed_lemmatized"].apply(contains_migration)
]

In [ ]:
df_afd_2018_migration["text_preprocessed_lemmatized"].tolist()[296]

'verehrter linke abschiebung rueckfuehrung moegen diplomatie freiwillige rueckkehr moegen syrien ungefaehr syrer genehmigung zitat veroeffentlichen beruehren essen kleidung geld fuehlen einsam ammar maarawi einzige zurueckwill angabe unhcr kehren haelfte syrische binnenfluechtlinge heimatstaedte fluechtling ausland syrien syrische aussenminister sicher rueckkehr garantieren russland syrien wiederaufnahmekapazitaeten befinden wiederaufbau andrej mahecic sprecher unhcr fluechtling rueckkehr rueckkehr wuerdevoll nachhaltig sechs plan sicher rueckkehr freiwillig rueckkehrer syrien dame abkomme syrien sicherstellen rueckkehrer unbeschadet syrien einreisen aufnehmen befriedet rueckkehrer aktivitaet flucht straftat verstoss militaerdienst verfolgen fliehen un wirksam ueberpruefen mensch freiwillig zurueckwollen mensch assad moegen herrschaft assads fakt schreien herrschaft assads fakt umgehen irgendwann rueckkehr fluechtling thematisieren fluechtling kehren unsicher abkomme kontrollieren hinw

'verehrter linke abschiebung rueckfuehrung moegen diplomatie freiwillige rueckkehr moegen syrien ungefaehr syrer genehmigung zitat veroeffentlichen beruehren essen kleidung geld fuehlen einsam ammar maarawi einzige zurueckwill angabe unhcr kehren haelfte syrische binnenfluechtlinge heimatstaedte fluechtling ausland syrien syrische aussenminister sicher rueckkehr garantieren russland syrien wiederaufnahmekapazitaeten befinden wiederaufbau andrej mahecic sprecher unhcr fluechtling rueckkehr rueckkehr wuerdevoll nachhaltig sechs plan sicher rueckkehr freiwillig rueckkehrer syrien dame abkomme syrien sicherstellen rueckkehrer unbeschadet syrien einreisen aufnehmen befriedet rueckkehrer aktivitaet flucht straftat verstoss militaerdienst verfolgen fliehen un wirksam ueberpruefen mensch freiwillig zurueckwollen mensch assad moegen herrschaft assads fakt schreien herrschaft assads fakt umgehen irgendwann rueckkehr fluechtling thematisieren fluechtling kehren unsicher abkomme kontrollieren hinweis echauffieren sparen moralkeule deal despoten erdogan saudi arabien assad rueckfuehrungsabkommen abschiebemoeglichkeit schliessen merkel dame heucheln mensch freiwillige rueckkehr syrisch fluechtling ehrlichkeit menschlichkeit fluechtling schuldig herzliche dank'

In [ ]:
df_afd_2018_migration["text"].tolist()[300]

'Sehr geehrter Herr Präsident! Werte Kolleginnen und Kollegen! Staaten, die nicht bereit sind, ihren eigenen Staatsbürgern Papiere auszustellen und so die Rückführungen ihrer eigenen Staatsbürger verhindern, muss die Entwicklungshilfe gestrichen werden.\n\nDas sagt nicht nur der gesunde Menschenverstand; das fordern wir in unserem Antrag.\n\nNach Auskunft des Bundesinnenministeriums halten sich in Deutschland über eine halbe Million Menschen mit einem abgelehnten Asylantrag auf. Nicht wenige davon können aufgrund fehlender Ausweisdokumente nicht abgeschoben werden. Sind Ausweisdokumente nicht vorhanden, dann können biometrische Daten weiterhelfen.\n\nIch habe Minister Müller bereits im Bundestag dazu befragt. Ich zitiere mit Erlaubnis des Präsidenten die Antwort des Herrn Minister:\n\nZitat Ende.\xa0– Erstaunlich fand ich übrigens, dass dieser Umstand die Kollegen der Unionsfraktionen laut Plenarprotokoll erheitert hat. Was gibt es da eigentlich zu lachen, wenn ein Entwicklungsland sch

'Sehr geehrter Herr Präsident! Werte Kolleginnen und Kollegen! Staaten, die nicht bereit sind, ihren eigenen Staatsbürgern Papiere auszustellen und so die Rückführungen ihrer eigenen Staatsbürger verhindern, muss die Entwicklungshilfe gestrichen werden.\n\nDas sagt nicht nur der gesunde Menschenverstand; das fordern wir in unserem Antrag.\n\nNach Auskunft des Bundesinnenministeriums halten sich in Deutschland über eine halbe Million Menschen mit einem abgelehnten Asylantrag auf. Nicht wenige davon können aufgrund fehlender Ausweisdokumente nicht abgeschoben werden. Sind Ausweisdokumente nicht vorhanden, dann können biometrische Daten weiterhelfen.\n\nIch habe Minister Müller bereits im Bundestag dazu befragt. Ich zitiere mit Erlaubnis des Präsidenten die Antwort des Herrn Minister:\n\nZitat Ende.\xa0– Erstaunlich fand ich übrigens, dass dieser Umstand die Kollegen der Unionsfraktionen laut Plenarprotokoll erheitert hat. Was gibt es da eigentlich zu lachen, wenn ein Entwicklungsland schafft, was die Bundesregierung nicht kann oder nicht will?\n\nLieber Herr Minister Müller\xa0– vom BMZ ist ja, glaube ich, niemand da\xa0–, wenn Sie das nächste Mal Hunderte Millionen Euro nach Marokko oder Tunesien überweisen, dann tun Sie doch den Kollegen Innenministern einen Gefallen und verlangen Sie die biometrischen Daten. Die IBAN scheint ja zu funktionieren. Zur Not schreiben Sie Ihr Anliegen in den Verwendungszweck. Da haben Sie 140\xa0Zeichen.\n\nDie Bundesregierung hat bereits 2016 in einem Akt der Verzweiflung Brandbriefe an insgesamt 17\xa0Staaten verschickt: Ägypten, Algerien, Marokko, Äthiopien, Benin, Burkina Faso, Ghana, Guinea, Guinea-Bissau, Mali, Niger, Nigeria, Tunesien, Bangladesch, Indien, Pakistan und Libanon. Und hat das Briefeschreiben was gebracht? Offensichtlich nicht.\n\nLiebe Freunde der Regierungsfraktionen, gemeinsam mit uns, der AfD, kann dieses Problem heute gelöst werden. Machen Sie heute zur Abwechslung einfach mal das Richtige!\n\nWillensbekundungen in die Richtung gab es von Ihnen schon zuhauf. Sachsens Ministerpräsident Michael Kretschmer sagte der „Frankfurter Allgemeinen Sonntagszeitung“ noch im Mai:\n\nBayerns Noch-Innenminister Joachim Herrmann sagte, dass man manchmal über die Entwicklungshilfe Druck auf Herkunftsländer machen müsse. Anfang\xa02016 ging sogar der damalige SPD-Chef Sigmar Gabriel mit der Idee hausieren, man werde nordafrikanischen Staaten die Entwicklungshilfe kürzen, wenn sie Illegale ohne Aufenthaltsrecht nicht zurücknehmen. Liebe Freunde von der SPD, da können Sie ruhig schon mal klatschen. Der Mann war immerhin Ihr Vorsitzender zu einer Zeit, als Sie noch nicht in den Umfragen hinter der AfD lagen.\n\nWas haben denn diese Herren alle gemein? Sie sind seit vielen, vielen Jahren in Regierungsverantwortung. Und was haben sie bisher gemacht? Sie haben nichts gemacht.\n\nIn einer Infratest-dimap-Umfrage vom Frühling dieses Jahres sprechen sich 59\xa0Prozent der Bürger für die Kürzung von Entwicklungshilfe bei nichtkooperativen Ländern aus. Unser Antrag sieht aber nicht einmal einen sofortigen radikalen Schnitt vor,\n\nsondern setzt auf mehrere Eskalationsstufen bis hin zur völligen Streichung der Mittel. Sie haben deshalb die einmalige Chance, Ihre Glaubwürdigkeit ein Stück weit zu reparieren. So eine Chance bekommt man nicht jeden Tag.\n\nSie wissen selbst, was passiert, wenn man zu viel \xadseehofert, also Dinge verspricht und dann nicht hält. Sie haben es am Sonntag in Bayern gesehen,\n\nund sie werden es nächste Woche Sonntag in Hessen wieder sehen. Nutzen Sie heute die Chance: Machen Sie endlich mal das Richtige!\n\nVielen Dank.'


### Examining migration speech data from 2019

In [ ]:

df_afd_2019 = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2019)
].copy()


df_afd_2019_migration = df_afd_2019[
    df_afd_2019["text_preprocessed_lemmatized"].apply(contains_migration)
]

In [ ]:
df_afd_2019_migration["text_preprocessed_lemmatized"].tolist()[195]

'dame minute fange starten sinngemaess lucassen ausfuehrung verteidigungshaushalt rechtsstaat allgemeine erodiert druck geraten rottmann justizministerium besondere desolat zustand beschreibung bundeswehr einbeziehung justizministeriums abschliessen nahezu befinden altparteien dilcher merkel desolat zustand schlimm ddr sozialismus schlecht dorthin generaldebatte glueck einzelplan beschraenken dame etat gesamtetat kleinste vernachlaessigenswert ehemals stolze justizressort martens fuehrung durchlauferhitzer zwischenzulagernde karriereristen verkommen heiko maas schraegen justizminister schlimm dilettiert bekanntlich aussenministerium katarina barley naechst rohrkrepierer naechst rohrkrepiererin amt erzaehlen stichwort pakt rechtsstaat stichwort anwaltsverguetungen laecheln regeln weggelobt fluechten entsorgen eu schlimm lambrecht willkommen naechst sitzungswoche monat amtsantritt stunde rechtsausschuss schenken lambrecht strecken rechte hinten terminkalender schieben missachtung rechtsa

'dame minute fange starten sinngemaess lucassen ausfuehrung verteidigungshaushalt rechtsstaat allgemeine erodiert druck geraten rottmann justizministerium besondere desolat zustand beschreibung bundeswehr einbeziehung justizministeriums abschliessen nahezu befinden altparteien dilcher merkel desolat zustand schlimm ddr sozialismus schlecht dorthin generaldebatte glueck einzelplan beschraenken dame etat gesamtetat kleinste vernachlaessigenswert ehemals stolze justizressort martens fuehrung durchlauferhitzer zwischenzulagernde karriereristen verkommen heiko maas schraegen justizminister schlimm dilettiert bekanntlich aussenministerium katarina barley naechst rohrkrepierer naechst rohrkrepiererin amt erzaehlen stichwort pakt rechtsstaat stichwort anwaltsverguetungen laecheln regeln weggelobt fluechten entsorgen eu schlimm lambrecht willkommen naechst sitzungswoche monat amtsantritt stunde rechtsausschuss schenken lambrecht strecken rechte hinten terminkalender schieben missachtung rechtsausschusses ausdruecken plan amt lambrecht monat amt verdaddelt schwerpunkt lagen absolution schulschwaenzer zustaendig hektisch unausgegoren ankuendigung waffenrecht zustaendig ansonsten museum rechtsstaat aeussern ressort lambrecht visitenkarte rechtsstaats gewaltmonopol unabhaengig justiz rechtsstaat lambrecht bundesjustiz verfassungsministerin millionenfacher rechtsbruch fehlend rechtsdurchsetzung ueberwiegen illegal migration aeussern lambrecht richter gericht flut importiert kriminalitaet million straftat zuwanderer ueberrollen aeussern lambrecht lambrecht hunderttausende rechtsstaatswidrig vollzogen abschiebung aeussern lambrecht zigtausende vollzogen haftbefehle aeussern lambrecht staerkung rechtsstaats pakt rechtsstaat aeussern lambrecht erosion rechtsstaats zunehmend durchbrechung gewaltenteilung auswahl ober richter bundesverfassungsrichter parteibuch parteienkluengel bruch europarechts aeussern lambrecht stunde lambrecht charmante gesicht exekutive rechtsstaat stolz anwalt rechtsstaats stoppen erosion rechtsstaats dank'

In [ ]:
df_afd_2019_migration["text"].tolist()[200]

'Sehr geehrte Frau Präsidentin! Werte Kollegen! Vergeblich haben wir, die AfD, immer wieder Statistiken und Auswertungen zu flüchtlingsbedingten Kosten im Zusammenhang mit dem Gesundheitsfonds gefordert.\n\nGeschehen ist nichts und kann es auch derzeit nicht. Denn es gibt keine aussagefähigen Zahlen, die als Grundlage für den Gesundheitsfonds dienen können.\n\nWoran liegt das? Diese Große Koalition hat jahrelang zugelassen, dass Krankenkassen und Krankenhäuser unzulässige pauschale Rechnungskürzungen\xa0– wahrscheinlich insgesamt in Höhe mehrerer Milliarden\xa0– vereinbaren und auf diese Weise Abrechnungsprüfungen umgehen. Das ist ein bodenloser Skandal auf Kosten der Beitragszahler, der Krankenkassen und des Steuerzahlers.\n\nDer Bundesrechnungshof und das Bundesversicherungsamt und die Aufsichtsbehörden der Länder haben dann auch endlich im November 2018 die Rechtswidrigkeit dieser Vereinbarungen ausdrücklich bestätigt. Über Jahre kamen Krankenkassen ihrer Pflicht zur Prüfung der Kra

'Sehr geehrte Frau Präsidentin! Werte Kollegen! Vergeblich haben wir, die AfD, immer wieder Statistiken und Auswertungen zu flüchtlingsbedingten Kosten im Zusammenhang mit dem Gesundheitsfonds gefordert.\n\nGeschehen ist nichts und kann es auch derzeit nicht. Denn es gibt keine aussagefähigen Zahlen, die als Grundlage für den Gesundheitsfonds dienen können.\n\nWoran liegt das? Diese Große Koalition hat jahrelang zugelassen, dass Krankenkassen und Krankenhäuser unzulässige pauschale Rechnungskürzungen\xa0– wahrscheinlich insgesamt in Höhe mehrerer Milliarden\xa0– vereinbaren und auf diese Weise Abrechnungsprüfungen umgehen. Das ist ein bodenloser Skandal auf Kosten der Beitragszahler, der Krankenkassen und des Steuerzahlers.\n\nDer Bundesrechnungshof und das Bundesversicherungsamt und die Aufsichtsbehörden der Länder haben dann auch endlich im November 2018 die Rechtswidrigkeit dieser Vereinbarungen ausdrücklich bestätigt. Über Jahre kamen Krankenkassen ihrer Pflicht zur Prüfung der Krankenhausabrechnungen nicht nach. Sie hatten individuelle Vereinbarungen mit den Krankenhäusern über pauschale Rechnungskürzungen beschlossen und im Gegenzug auf Abrechnungsprüfungen verzichtet. Damit unterblieben auch die für bestimmte Fälle gesetzlich vorgeschriebenen Prüfungen durch den Medizinischen Dienst der Krankenversicherung. Krankenkassen sind aber gesetzlich verpflichtet, eine gutachtliche Stellungnahme des Medizinischen Dienstes der Krankenversicherung einzuholen, wenn dies nach Art, Schwere, Dauer oder Häufigkeit der Erkrankung oder nach dem Krankheitsverlauf erforderlich ist.\n\nMit anderen Worten: Diese Vereinbarungen ermöglichen es Krankenhäusern, sich von Prüfungen durch die Krankenkassen und damit des Medizinischen Dienst freizukaufen. Wer aber hindert die Krankenhäuser, die Abzüge im Vorfeld einzukalkulieren und überhöhte Rechnungen auszustellen, vor allem, wenn sie wissen, dass eine Überprüfung ohnehin nicht stattfindet? Es wurde also ein System erschaffen und von dieser Bundesregierung jahrelang geduldet, das millionenfache Gelegenheit zum Abrechnungsbetrug ermöglicht.\n\nEs ist ein System, in dem durch pauschalen Verzicht der Überprüfung der Krankenhausabrechnungen in Kauf genommen wurde, dass medizinische Fälle unerkannt bleiben, in denen der Medizinische Dienst eingeschaltet werden müsste. Mit anderen Worten: Derzeit haben weder die Krankenkassen noch die Bundesregierung einen genauen Überblick, mit welchen Krankheiten unsere Bevölkerung in welchem Umfang wirklich zu tun hat\xa0\nein Skandal und eine kaum zu übertreffende Verantwortungslosigkeit jedem einzelnen Bürger unseres Landes gegenüber.\n\nUnd es geht weiter: Da die so gewonnenen Daten die Grundlage für die Zuweisung aus dem Gesundheitsfonds bilden, sind alle Berechnungen, die dem Gesundheitsfonds derzeit zugrunde liegen, unbrauchbar, Makulatur, schlicht und einfach für die Katz.\n\nUnd was macht diese Bundesregierung? Sie lässt sich Zeit. Erst am 17.\xa0Juli 2019, also in der Sommerpause, wurde vom Kabinett der Entwurf eines Reformgesetzes beschlossen, um eine gesetzliche Klarstellung des Verbots dieser im November 2018 endlich als unzulässig erkannten Vereinbarung in die Wege zu leiten. Jahrealte Abrechnungen müssen jetzt überprüft werden, um Abrechnungsfehler und vor allem Betrügereien zu entdecken, die zu einem Schaden von vielen Milliarden Euro geführt haben können.\n\nWie viele Millionen kostet diese Aufarbeitung den Steuerzahler\xa0– denn wer sonst soll das alles bezahlen? Wie können Sie von Bürgern Gesetzestreue erwarten, wenn Sie selbst so grob fahrlässig mit der Gesundheit und dem Geld unserer Bürger umgehen? Wie können Sie bei dieser Sach- und Rechtslage zuverlässige Aussagen über den Finanzbedarf des Gesundheitsfonds treffen?\n\nLast, but not least: Nach den neuesten Zahlen des Robert-Koch-Instituts sind Erkrankungen an Hepatitis B im letzten Jahr weiter explosiv gestiegen. Hepatitis B zählt bei chronischem Verlauf zu den bedeutendsten Ursachen von Leberzellkarzinomen,\n\nund der Tod als Folge hiervon rangiert weltweit auf Platz zwei der krebsbedingten Todesursachen.\n\nRechnet man die diesjährigen Zahlen des RKI hoch, kommt man auf 5\xa0560 Fälle. Das bedeutet von 2018 zu 2019 eine Steigerung von circa 20\xa0Prozent, also eine Steigerung von circa 635\xa0Prozent seit 2014, als es in Deutschland nur 755\xa0Fälle gab. Bei Asylsuchenden kamen 2017\xa0\xa062\xa0Prozent aus Afrika und 29\xa0Prozent aus Asien, vorwiegend aus Syrien und Afghanistan.\n\nAktuelle Studien aus Deutschland zeigen laut RKI für Personen mit Migrationshintergrund, dass 80\xa0Prozent ihrer Erkrankungen an einer aktiven Hepatitis\xa0B unbekannt war und dass sie auch nicht wussten, wie Hepatitis\xa0B übertragen wird.\n\nWir, die AfD, fordern daher gezielte Screeningmaßnahmen bei Asylsuchenden;\n\ndenn die Dunkelziffer der Infizierten birgt wie bei HIV und Tuberkulose ein schreckliches epidemiologisches Potenzial.\n\nOder sollen solche Fälle wie der im September 2017 in Dresden gehäuft auftreten, Herr Spahn, wo nach der Entdeckung eines aktiven TB-Falls circa 2\xa0000 Kontaktpersonen ermittelt wurden und über 3\xa0000 Blutentnahmen erfolgt sind? Damals wurden 120 latente tuberkulöse Infektionen und sieben aktive Tuberkulosen ermittelt.\n\nWir, die AfD, brechen diese Schweigespirale. Unsere Bevölkerung muss vor derartigen Krankheiten geschützt werden, und das nicht nur wegen der exorbitant hohen Kosten.\n\nUnd wie oft müssen wir, die AfD, noch auf diese für jedermann offenkundigen Gefahren hinweisen, bis auch Sie, die Sie hier alle sitzen, endlich handeln?'












































































### Examining migration speech data from 2020

In [ ]:

df_afd_2020 = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2020)
].copy()


df_afd_2020_migration = df_afd_2020[
    df_afd_2020["text_preprocessed_lemmatized"].apply(contains_migration)
]

In [ ]:
df_afd_2020

,speech_identification_ent,date,period,session,pos_speechbeginning,Party,Role,governing_Party,text,text_length,text_preprocessed,text_preprocessed_lemmatized,text_length_preprocessed,text_length_lemmatized,date_year,date_quarter,similarity_expansive,similarity_restrictive,periode_index,modified_flag
223397,René Springer,2020-01-15,19,139,NaN,AfD,MdB,0.0,"Herr Minister, danke für die Neujahrswünsche. ...",122,neujahrswuensche eu eu mitgliedstaaten unterbr...,neujahrswuensche gute eu eu mitgliedstaaten un...,27,32,2020,1,0.140278,0.137391,2020Q1,False
223399,René Springer,2020-01-15,19,139,NaN,AfD,MdB,0.0,"Herr Minister, völlig überraschend haben Sie d...",132,ueberraschend beantwortet konkreter plan eu ra...,ueberraschen frage plan eu ratspraesidentschaf...,25,24,2020,1,0.017772,0.092324,2020Q1,False
223413,Ulrike Schielke-Ziesing,2020-01-15,19,139,NaN,AfD,MdB,0.0,"Guten Tag, Herr Minister! Wie wir den Medien e...",132,medien entnehmen kabinettsbeschluss grundrente...,guten medium entnehmen naechst kabinettsbeschl...,34,37,2020,1,0.162233,0.147685,2020Q1,False
223415,Ulrike Schielke-Ziesing,2020-01-15,19,139,NaN,AfD,MdB,0.0,"Es geht mir ja nicht darum, dass wir ein autom...",123,automatisiertes akten worst case szenario auto...,automatisiert akte worst case szenario million...,22,24,2020,1,0.307247,0.274563,2020Q1,False
223435,Uwe Witt,2020-01-15,19,139,NaN,AfD,MdB,0.0,"Liebe Kollegen, ich wünsche uns allen ein froh...",149,wuensche frohes neues ereignisse demnaechst zu...,lieben wuensche ereignis demnaechst zumarschie...,31,30,2020,1,0.060526,0.015937,2020Q1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230272,Gerold Otten,2020-12-17,19,202,NaN,AfD,MdB,0.0,Herr Präsident! Sehr geehrte Kolleginnen und K...,674,emotionale mehreren foren bewaffneten drohnen ...,emotionale monat mehrer forum bewaffnen drohne...,206,196,2020,4,-0.040217,-0.030218,2020Q4,False
230278,Uwe Schulz,2020-12-17,19,202,NaN,AfD,MdB,0.0,Frau Präsidentin! Meine Damen und Herren! Weih...,665,weihnachtsstimmung langsam ausbau netzes fahrt...,dame weihnachtsstimmung langsam ausbau netzes ...,181,178,2020,4,-0.132343,-0.163622,2020Q4,False
230286,Norbert Kleinwächter,2020-12-17,19,202,NaN,AfD,MdB,0.0,Werte Frau Präsidentin! Sehr geehrte Kolleginn...,648,werte nachholfaktor rentenversicherung einfueh...,nachholfaktor rentenversicherung einfahren ren...,175,170,2020,4,0.107760,0.086961,2020Q4,False
230294,Nicole Höchst,2020-12-17,19,202,NaN,AfD,MdB,0.0,Frau Präsidentin! Werte Kollegen! Liebe Bürger...,510,werte beinahe film taeglich gruesst murmeltier...,lieben beinahe film taeglich gruessen murmelti...,169,171,2020,4,0.041223,0.029622,2020Q4,False


In [ ]:
df_afd_2020_migration["text_preprocessed_lemmatized"].tolist()[286]

'dame lieben zuschauer bildschirmen einzelplan umfangreiche haushaltsberatungen umfassten inner staerken asyl migration massiv absenken sport schwierig neueinstellungen sicherheitsbehoerden eklatanten versaeumnis vorgaengerregierungen nachholen afd modernisierung sicherheitsstruktur modern ausgebildet grenze innere material personal aufstocken seehofer aufstocken wille ursache gewachsen bedrohungslage effizienter grenzschutz asylmissbrauch importieren kriminalitaet fehlanzeige islamistische gefaehrder grotesk aufwand ueberwachen anstatt komplett abschiebehaft praeventivgewahrsam unfassbar teuer staatlich alimentierung linksextremisten aufsetzen bekaempfung gewaltbereiten islamismus verstaerken bekaempfung jeglich extremismus genehmen demonstration polizei diffamieren bejubeln einsatz wasserwerfern reizgas demo einschraenkung grundrechte schaden ansehen ordnungskraefte fakt polizei spielball rueckhalt verlieren rueckendeckung seehofer heimat begriff bieten heimatschutz massnahme aktiv v

'dame lieben zuschauer bildschirmen einzelplan umfangreiche haushaltsberatungen umfassten inner staerken asyl migration massiv absenken sport schwierig neueinstellungen sicherheitsbehoerden eklatanten versaeumnis vorgaengerregierungen nachholen afd modernisierung sicherheitsstruktur modern ausgebildet grenze innere material personal aufstocken seehofer aufstocken wille ursache gewachsen bedrohungslage effizienter grenzschutz asylmissbrauch importieren kriminalitaet fehlanzeige islamistische gefaehrder grotesk aufwand ueberwachen anstatt komplett abschiebehaft praeventivgewahrsam unfassbar teuer staatlich alimentierung linksextremisten aufsetzen bekaempfung gewaltbereiten islamismus verstaerken bekaempfung jeglich extremismus genehmen demonstration polizei diffamieren bejubeln einsatz wasserwerfern reizgas demo einschraenkung grundrechte schaden ansehen ordnungskraefte fakt polizei spielball rueckhalt verlieren rueckendeckung seehofer heimat begriff bieten heimatschutz massnahme aktiv vorangetrieben haushalt miete ballungsraeumen polizeiabsolventen zwingen wohngemeinschaft bilden anfahrtswege dienststellen kauf jung absolventen grossstadt ortszuschlag einfuehren wohnraum bundesbedienstete bestand zustaendig wohnfuersorge bundesfinanzministerium abschieben wohnungsbau steigerung anstrengung abschluss ueppig ausstattung stiftung verwundern bereinigungssitzung globalzuschuesse stiftung million erhoehen nullen haushaltsausschuss mittelbedarf zustande nullen ueppig mittelausstattung stiftung intransparenter prozess mitwirken profitieren lehnen haushalt'



In [ ]:
df_afd_2020_migration["text"].tolist()[303]

'Ich bemühe mich, Frau Präsidentin.\xa0– Frau Präsidentin! Werte Kolleginnen und Kollegen! Wir sprechen heute über einen verbesserten Zugang zu Teilhabeleistungen. Das sind Leistungen, die Menschen mit Behinderungen dabei helfen, ihr Leben zu meistern.\n\nSo wichtig es ist, über bessere Regelungen in der Teilhabe zu reden, so wichtig ist es, dabei fair und gerecht zu bleiben. Menschen mit Behinderungen haben es verdient, dass sie nicht zum Spielball von Ideologie werden, sondern dass die Politik ihre Bedürfnisse ernst nimmt. Und darum geht es Ihnen von den Grünen leider nicht.\n\nFrau Rüffer, was Sie vorgetragen haben, ist unglaubwürdig, wenn man das mit dem Antrag vergleicht, den Sie gestellt haben.\n\nDer Antrag bringt keine echten Lösungen; er fabuliert an verschiedenen Stellen, dass etwas verbessert werden soll, sagt aber nicht, wie. Vieles von dem, was Sie vorschlagen, existiert schon. Was ich Ihnen wirklich vorwerfe, ist die Tatsache, dass es keinen einzigen Antrag der Grünen zu 

'Ich bemühe mich, Frau Präsidentin.\xa0– Frau Präsidentin! Werte Kolleginnen und Kollegen! Wir sprechen heute über einen verbesserten Zugang zu Teilhabeleistungen. Das sind Leistungen, die Menschen mit Behinderungen dabei helfen, ihr Leben zu meistern.\n\nSo wichtig es ist, über bessere Regelungen in der Teilhabe zu reden, so wichtig ist es, dabei fair und gerecht zu bleiben. Menschen mit Behinderungen haben es verdient, dass sie nicht zum Spielball von Ideologie werden, sondern dass die Politik ihre Bedürfnisse ernst nimmt. Und darum geht es Ihnen von den Grünen leider nicht.\n\nFrau Rüffer, was Sie vorgetragen haben, ist unglaubwürdig, wenn man das mit dem Antrag vergleicht, den Sie gestellt haben.\n\nDer Antrag bringt keine echten Lösungen; er fabuliert an verschiedenen Stellen, dass etwas verbessert werden soll, sagt aber nicht, wie. Vieles von dem, was Sie vorschlagen, existiert schon. Was ich Ihnen wirklich vorwerfe, ist die Tatsache, dass es keinen einzigen Antrag der Grünen zu geben scheint, in dem nicht irgendwo Politik für Migranten versteckt ist.\n\nIn diesem spezifischen Antrag fabulieren Sie sogar Menschenrechte herbei, die es nicht gibt, um Politik gegen Deutschland zu begründen. Das ist in der Behindertenpolitik besonders verwerflich, meine Damen und Herren!\n\nUnd darauf will ich eingehen: Auf Seite\xa08 ersinnen Sie, das Recht auf Teilhabe sei ein universelles Menschenrecht, um dann gleich hinzuzufügen, dass es nicht vom aufenthaltsrechtlichen Status abhängig gemacht werden darf.\n\nDamit begründen Sie ein unbegrenztes Leistungs-, Wunsch- und Wahlrecht. Die Gemeinschaft hat alles zu bezahlen für alle aus aller Welt. Die Wahrheit ist aber\xa0– um das jetzt ein für alle Mal zu klären\xa0–: Ein universelles Menschenrecht auf Teilhabe gibt es nicht, übrigens genauso wenig wie auf Sozialleistungen oder Migration,\n\nnicht in der Europäischen Menschenrechtskonvention und auch nicht in der Allgemeinen Erklärung der Menschenrechte der UN. Dort gibt es aber eine gegenseitige Verantwortlichkeit zwischen Gemeinschaft und Individuum. Lesen Sie mal Artikel\xa022: Als Mitglied der Gesellschaft hat jeder unter Berücksichtigung der Mittel des Staates Anspruch darauf, in den Genuss der Rechte zu gelangen, die für seine Würde unentbehrlich sind.\xa0– Auf der anderen Seite gibt es Artikel\xa029: „Jeder hat Pflichten gegenüber der Gemeinschaft.“\xa0– Ja, schreiben Sie sich das mal hinter die grünen Ohren!\n\nUnd in der UN-Behindertenrechtskonvention gibt es auch keine entsprechenden Regelungen. In Artikel\xa019 haben die Vertragsstaaten deklariert, dass sie Maßnahmen treffen, um Menschen mit Behinderungen ihre volle Einbeziehung in die Gemeinschaft und Teilhabe an der Gemeinschaft zu erleichtern. Das steht aber unter dem Progressionsvorbehalt des Artikels\xa04, nämlich der verfügbaren Mittel und der Verwirklichung nach und nach.\n\nDer Anspruch auf Leistungen ist also kein universelles Menschenrecht. Der Staat hat die Pflicht, im Rahmen seiner Mittel für jedes Mitglied der Gesellschaft auf Teilhabe hinzuwirken. Ihre Prämisse, die Sie in den Antrag geschrieben haben, ist einfach falsch, wie in vielen anderen Anträge auch. Und das werfe ich Ihnen vor: dass Sie die Behindertenpolitik zu dieser Art von Forderungen missbrauchen.\n\nDie anderen wohlfeilen Forderungen, die Sie stellen, fallen ganz schnell auseinander, wenn man sich intensiver damit beschäftigt. Ich mache mal einen kurzen Durchlauf.\n\nErstens. Das Wunsch- und Wahlrecht wollen Sie ausweiten, insbesondere bezüglich der Leistungen und des Wohnorts und des Pflegetyps. Nach der geltenden Rechtslage gibt es bereits jetzt einen Ausschluss deutlich teurerer Leistungen nur nach einer Abwägung der Situation, nach einer Prüfung der Zumutbarkeit. Das steht in §\xa0104 SGB\xa0IX; den muss man halt lesen.\n\nZweitens. Sie wollen neue Berichtspflichten und neue Fristen einführen. Sie haben selber festgestellt, dass die Behörden schon überlastet sind; Sie wollen sie aber noch weiter überfordern, obwohl es solche Fristen schon längst gibt. Bei Überschreitungen gibt es die sogenannte Genehmigungsfiktion; das heißt, die Leistungsberechtigten könnten sich die Leistungen einfach selber einkaufen. Das stört Sie aber wieder. Sie sagen, das könnten die nicht finanzieren, haben aber offensichtlich überlesen, dass diese Leute Anspruch auf Abschlagszahlungen haben, und zwar nach §\xa018 Absatz\xa04 SGB\xa0IX.\n\nGanz wichtig sind Ihnen natürlich die Leistungen für Asylbewerber. Sie fordern Eingliederungshilfen vom ersten Tag an. Wollen wir wirklich den eingliedern, der geduldet ist oder direkt vor der Abschiebung steht?\xa0– Das ist ganz bestimmt nicht das Ziel von Behindertenrechtspolitik.\n\nMeine Damen und Herren, der Antrag, den Sie gestellt haben, dient nicht den Menschen mit Behinderungen. Er dient dazu, die Tür zu öffnen für Eingliederung und Extraleistungen für Menschen aus der ganzen Welt. Teilhabe wird so erweitert zu einer Art Nachteilsausgleich für illegale Migranten. Wenn Sie so die Menschen mit Behinderungen für Ihre One-World-Ideologie\xa0–'























### Examining migration speech data from 2022

In [ ]:

df_afd_2022 = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2022)
].copy()


df_afd_2022_migration = df_afd_2022[
    df_afd_2022["text_preprocessed_lemmatized"].apply(contains_migration)
]

In [ ]:
df_afd_2022_migration["text_preprocessed_lemmatized"].tolist()[248]

'dame krankenhauspflege erschreckend jahrelang ignoranz herrschend fundamental pflegemassnahmen geboten aufwand durchfuehren personal ausbildung mangeln impliziten rationierung pflege patient ueberwachung patient gespraech angehoerige korrekt dokumentation anmessen pflegearbeit zunehmen aushoehlen qualifizieren ausgebildet pflegekraefte uebernehmen personal gewaehrleisten krankenschwester kranke kuemmern hol bringedienst reinigungskraft hauptberufliche dokumentationskraft resultierend unzufriedenheit muenden flucht qualifiziert beruf ungerecht corona pflegebonusgesetz mitarbeiter leeren unmut pflege krankenhausbereich teufelskreis durchbrechen frage personal akquirieren vortrag ueblich zurufe links block zuwanderung heissen krankenhauspflege rot gruen zuwanderung psychologisch werte zuwanderung jahrzehnt kilimandscharo auftuermen attraktiv zuwanderungsland auslaendische fachkraefte steuer abgabe arbeitsbedingungen schlecht attraktivste zuwanderungsland sozialfluechtlinge europaeisch na

'dame krankenhauspflege erschreckend jahrelang ignoranz herrschend fundamental pflegemassnahmen geboten aufwand durchfuehren personal ausbildung mangeln impliziten rationierung pflege patient ueberwachung patient gespraech angehoerige korrekt dokumentation anmessen pflegearbeit zunehmen aushoehlen qualifizieren ausgebildet pflegekraefte uebernehmen personal gewaehrleisten krankenschwester kranke kuemmern hol bringedienst reinigungskraft hauptberufliche dokumentationskraft resultierend unzufriedenheit muenden flucht qualifiziert beruf ungerecht corona pflegebonusgesetz mitarbeiter leeren unmut pflege krankenhausbereich teufelskreis durchbrechen frage personal akquirieren vortrag ueblich zurufe links block zuwanderung heissen krankenhauspflege rot gruen zuwanderung psychologisch werte zuwanderung jahrzehnt kilimandscharo auftuermen attraktiv zuwanderungsland auslaendische fachkraefte steuer abgabe arbeitsbedingungen schlecht attraktivste zuwanderungsland sozialfluechtlinge europaeisch nachbar freund fuellen sonderzug mensch potenzial million zuwanderer genuegen fachkraefte sortierung koffer flughafen gewinnen geschweige ausgebildet pflegekraefte absolut unverzichtbar sprachkenntnissen dazugehoeren moegen explizit bundesweite studie pflegen arbeitnehmerkammer bremen befragte erwaehnen besagen vollzeitpflegekraefte rueckkehr beruf aufstockung arbeitszeit sofern arbeitsbedingungen pflege personal mangeln tausende einrichtungsbezogene impfpflicht beruf abschrecken taetigen einstellungsstau rechtswidrige einrichtungsbezogene impfpflicht laufen aussetzen gesundheits pflegewesen kernaufgaben begreifen unausweichlich pflegekatastrophe schlittern verantwortlich vorbildlich gesundheitswesen beschaedigen geld gesundheitssystem dank'


In [ ]:
df_afd_2022_migration["text"].tolist()[261]

'Sehr geehrte Frau Präsidentin! Sehr geehrte Damen und Herren! Herr Grötsch, die Bundespolizei nicht politisch zu instrumentalisieren – wir nehmen Sie und die Regierung beim Wort; immer wieder gern.\n\nDie Beschäftigung mit der inneren Sicherheit darf für die Politik niemals zum Selbstzweck werden. Vor fast sechs Jahren, am 19. Dezember 2016, tötete der polizei- und nachrichtendienstlich bekannte islamistische Attentäter Anis Amri auf dem Berliner Breitscheidplatz insgesamt 13 Menschen und verletzte 67 Personen. Solche Gräueltaten dürfen sich nie wieder wiederholen. In drei Tagen werden wir wieder in angemessener Form der Opfer dieser schrecklichen Tat gedenken. Ich bin in Gedanken bei den Verwandten der Opfer.\n\nEin unverzichtbarer Bestandteil des Schutzes der öffentlichen Sicherheit, aber tatsächlich auch des Schutzes der nationalen Grenzen, der Schienenwege und der Flughäfen ist unsere Bundespolizei. Circa 54\u202f000 Mitarbeiter sind in diesem Bereich tätig. Von 2017 bis 2021 hat 

'Sehr geehrte Frau Präsidentin! Sehr geehrte Damen und Herren! Herr Grötsch, die Bundespolizei nicht politisch zu instrumentalisieren – wir nehmen Sie und die Regierung beim Wort; immer wieder gern.\n\nDie Beschäftigung mit der inneren Sicherheit darf für die Politik niemals zum Selbstzweck werden. Vor fast sechs Jahren, am 19. Dezember 2016, tötete der polizei- und nachrichtendienstlich bekannte islamistische Attentäter Anis Amri auf dem Berliner Breitscheidplatz insgesamt 13 Menschen und verletzte 67 Personen. Solche Gräueltaten dürfen sich nie wieder wiederholen. In drei Tagen werden wir wieder in angemessener Form der Opfer dieser schrecklichen Tat gedenken. Ich bin in Gedanken bei den Verwandten der Opfer.\n\nEin unverzichtbarer Bestandteil des Schutzes der öffentlichen Sicherheit, aber tatsächlich auch des Schutzes der nationalen Grenzen, der Schienenwege und der Flughäfen ist unsere Bundespolizei. Circa 54\u202f000 Mitarbeiter sind in diesem Bereich tätig. Von 2017 bis 2021 hat sich die Anzahl der Planstellen bei der Bundespolizei von 42\u202f000 auf circa 50\u202f000 erhöht, der jährliche Haushalt ist im selben Zeitraum von 3,3 Milliarden auf 4,7 Milliarden Euro angehoben worden. Jeder einzelne Cent, den wir in unsere Bundespolizei investiert haben, ist gut angelegtes Geld für die Zukunft Deutschlands. Die Bundespolizei muss auch zukünftig weiter gestärkt und ausgebaut werden.\n\nAber es bleibt nach wie vor viel zu tun. Als Abgeordneter habe ich in den sitzungsfreien Wochen mehrfach Polizeidienststellen aufgesucht und mich mit Dienststellenleitern und Beamten des Polizeivollzugsdienstes unterhalten.\n\n– So fleißig sind wir. – Hierbei konnte ich mir ein eigenes Bild von den Gegebenheiten und den Anforderungen in den täglichen Abläufen bei der Bundespolizei machen. Einzelne Schwerpunkte sind dabei immer wieder zur Sprache gekommen:\n\nZur Bekämpfung der Schleuserkriminalität und zur Ahndung der Einreisestraftaten sollte es den Beamten zukünftig möglich sein, auch außerhalb der 30-Kilometer-Zone im gesamten Bundesgebiet eigenständig ermitteln zu dürfen.\n\nAuch die Endsachbearbeitung der von der Bundespolizei zuerst festgestellten Straftaten ist ein wichtiger Beitrag, um behördliche Ressourcen zu bündeln. Die derzeit noch gängige Praxis, eine Strafakte quasi für die Staatsanwaltschaft komplett vorzubereiten, um sie dann der Landespolizei zu übergeben, widerspricht modernen Effektivitätsgrundsätzen. Hier müssen dringend rechtliche Rahmenbedingungen geändert werden.\n\nDie Privatisierung der Autobahnen hat insbesondere in Grenzregionen dazu geführt, dass die Einrichtung von größeren Kontrollstellen nicht mehr unangekündigt möglich ist, sondern zuvor bei der privaten Betreibergesellschaft angemeldet werden muss. Die darauf folgenden Stauanzeigen auf den Navigationsgeräten der Fahrzeuge leiten den zu kontrollierenden Verkehr dann regelmäßig an der Kontrollstelle vorbei. Die Nutzung des Überraschungseffekts wird hierdurch deutlich eingeschränkt.\n\nEs bräuchte auch vermehrt sogenannte Verkehrstrichter, um die Geschwindigkeit des Fahrverkehrs bei derartigen Kontrollen des Verkehrs auf der Bundesautobahn zu verlangsamen. Eine solche Mittelbeschaffung wäre ein wichtiger Beitrag zum Arbeitsschutz unserer eingesetzten Beamten.\n\nBei für einen längeren Zeitraum eingerichteten Kontrollstellen, beispielsweise auf Autobahnen, benötigen die Beamten insbesondere an kalten Wintertagen, aber auch zum Schutz vor Regen und Nässe mobile Kontrollstellen mit entsprechender Ausstattung. Praxisbezogene Bedarfe sind beispielsweise sogenannte Agrarzelte, welche als Überdachung bei Kontrollen von Reisebussen erforderlich sind, damit die Fahrgäste und Beamten bei Wind und Wetter nicht im Freien stehen. Summa summarum benötigt die Bundespolizei zukünftig vermehrt lageangepasste Einsatzmittel, um Arbeitsweisen optimieren zu können.\n\nEines der größten Probleme im Dienstalltag der Bundespolizei bleibt aber die viel zu geringe Zahl an Rückführungen von Personen, die sich unerlaubt in Deutschland aufhalten. Im Jahr 2021 waren 33\u202f600 Rückführungen von Ausländern vorgesehen. Allerdings wurde davon nur die Hälfte, also rund 15\u202f000 Rückführungen, auch wirklich vollzogen. Der Jahresbericht der Bundespolizei sagt aus: Für die Diskrepanz an Rückführungen war hauptursächlich, dass die zur Abschiebung vorgesehenen Personen der Bundespolizei am Flugtag nicht zur Rückführung übergeben werden konnten. – Hier müssten Sie, Frau Innenministerin, ansetzen und endlich eine wirksame Abschiebeoffensive einleiten,\n\nanstatt Ihre Energie für die Jagd auf Phantombedrohungen zu verschwenden.\n\nWährend die Anzahl der im Jahr 2021 im Zuständigkeitsbereich der Bundespolizei festgestellten Straftaten in vielen Bereichen rückläufig war, lagen die größten Zunahmen im Bereich Betrug und bei Verstößen gegen das Aufenthaltsgesetz. Sage und schreibe 171\u202f000 Verstöße gegen das Aufenthaltsgesetz hat allein die Bundespolizei im Jahr 2021 festgestellt. Dies ist nicht zuletzt auf die Politik der offenen Grenzen unserer Bundesregierung zurückzuführen.\n\nLeider schweigt sich der Antrag der CDU dazu aus, wie Sie die genaue Zuständigkeitsverteilung bei der Strafverfolgung durch Land und Bund voneinander abgrenzen wollen. Eine Doppelzuständigkeit von Bundes- und Landespolizei zur Durchführung eines strafprozessualen Ermittlungsverfahrens etwa bei Taten, die an Bahnhöfen begangen werden, würde zu einem unüberschaubaren Behördenchaos führen. Das Thema Strafverfolgung ist aber zu bedeutend, als dass man es hier schaufensterartig in nur drei Zeilen abschließend darstellen könnte.\n\nZum Schluss sei noch gesagt: Beamte, insbesondere Polizeibeamte, sind Recht und Gesetz verpflichtet und dürfen nicht den politischen Fantasien der jeweiligen Regierung unterliegen. Nicht nur die Bundespolizisten gehören vor dem Generalverdacht geschützt, sondern auch neue Bewerber für den Polizeivollzugsdienst. Einen weiteren politischen Gesinnungs-TÜV von Nancy Faesers Gnaden lehnen wir entschieden ab.\n\nIch wünsche allen Deutschen einen friedlichen vierten Advent und frohe Weihnachten.'








































































### Examining migration speech data from 2024,2025

In [ ]:

df_afd_2024 = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year >2023)
].copy()


df_afd_2024_migration = df_afd_2024[
    df_afd_2024["text_preprocessed_lemmatized"].apply(contains_migration)
]

In [ ]:
df_afd_2024_migration["text_preprocessed_lemmatized"].tolist()[485]

'dame vordergruendig einbuergerung sonderfall option geringfuegig modifizierung vorausgegangen ausweitung zugriff staatsbuergerschaft ampel kuerzen frist einbuergerungsanspruch acht sonderfall zuvor fristenverkuerzung regelhafter akzeptanz doppelt staatsbuergerschaft heissen staatsbuergerschaft konsumartikel preis nullen absitzen warten turboeinbuergerung sonderfall beibehalten unfassbarer dreistigkeit taeuschen dame einbuergerung integriert integriert heissen hiesig leitkultur verinnerlichen heimat loyal stattdessen zerfallen zusehends regeln einbuergerung bleibt akzeptanz regelhaften doppeln staatsbuergerschaft bleibt generell einbuergerung frueh einbuergerung ermessen bleibt staatsbuergerschaft schleichen geburt frueh abstammung bleibt bleibt bleibt bleibt bleibt desaster totalversagen dame vollstaendig beibehaltung kosmetische korrektur absurd spezialregelung koalitionsvertrag durchsetzen unterwuerfig akzeptieren verkaufen grundsaetzliche wende einbuergerung behauptung unwahr wisse

'dame vordergruendig einbuergerung sonderfall option geringfuegig modifizierung vorausgegangen ausweitung zugriff staatsbuergerschaft ampel kuerzen frist einbuergerungsanspruch acht sonderfall zuvor fristenverkuerzung regelhafter akzeptanz doppelt staatsbuergerschaft heissen staatsbuergerschaft konsumartikel preis nullen absitzen warten turboeinbuergerung sonderfall beibehalten unfassbarer dreistigkeit taeuschen dame einbuergerung integriert integriert heissen hiesig leitkultur verinnerlichen heimat loyal stattdessen zerfallen zusehends regeln einbuergerung bleibt akzeptanz regelhaften doppeln staatsbuergerschaft bleibt generell einbuergerung frueh einbuergerung ermessen bleibt staatsbuergerschaft schleichen geburt frueh abstammung bleibt bleibt bleibt bleibt bleibt desaster totalversagen dame vollstaendig beibehaltung kosmetische korrektur absurd spezialregelung koalitionsvertrag durchsetzen unterwuerfig akzeptieren verkaufen grundsaetzliche wende einbuergerung behauptung unwahr wissentlich unwahr verdeckungspropaganda verhandlungsversagens gescheitert migrationswende verkaufen waehlertaeuschung dame verleihung staatsbuergerschaft ausweis angekommenseins heimat verstreichen integriertheit einbuergerung knapp million illegal eingedrungen migranten schwarz rot einbuergerungsanspruch schenken illegal migranten nahost afrika bestand verfolgung mehrheitlich nachbarland zig sicher vorteilsnahme mehrheitlich grossangelegten asylbetrug dame personenkreis angestammt kultur prosperitaet ueppig sozialsysteme kalten zyniker macht organisation per raschester einbuergerung antideutsche mangelnd qualifikation staatsabhaengiges prekariat importieren notgedrungen umverteilungsparteien waehlen einbuergerungspolitik aufforstungsprogramm rot rot gruene waehlerschaft wusste umfrage muslime auslaender angriff staatsvolk par excellence dame ansaessige langem kinderquote reproduktionsfaktor enkelgeneration halbieren gegenwaertig aktivierende familienpolitik unterlassen demografische katastrophe bevoelkerungstransformation zwingen verfestigen quasiautomatische einbuergerung zeitablauf sawsan chebli zitat demografie faktum erdogan zitat macht kind anschlag staatsvolk feindlich uebernahme dame vorwandcharakter operation tarnname fluechtlingsschutz klarname auslaenderimport absurd gehaeuft inkaufnahme nachteil aufnahmeland zweistellige milliardenbetraege veranstaltung kollaps wohnungsmarkts zusammenbruch bildungssystems totalverlust inner aufnahmeanspruch mehrheitlich gering durchzug zig sicher drittstaaten asyl subsidiaerer status assad buergerkrieg million syrer einbuergern dame merkel spiel erfinden ampel steigern gefordert einbuergerungsautomatismus illegal millionenheeres laufen friedrich merz linke vorbei friedrich merz ehrenwert merz kuendigen unumstoesslich amtszeit ausnahmslos illegal einreise zurueckweisen person schutzanspruch monat gibt asylbewerbern zweimal zurueckweisungen friedrich merz ehrenwert merkel moralisch rueckgrat brechen machtgeiler opportunismus friedrich merz schuldenmacherei vorbei friedrich merz ehrenwert verscherbeln anbiedern staatsbuergerschaft einbuergerung ueberproportionale auslaenderkriminalitaet statistisch unsichtbar einbuergerung abschiebung dauerhaft zugang sozialsystemen hinwendung einbuergerung per fristablauf dame lebensgefuehl mensch freibad kita schule endlos gemobbt nachts bahnhof zerstoeren innenstaedte staatsbuergerschaftsrecht ausschliesslich linke unumkehrbar identitaet zerstoeren hauptsache cduler kanzler kind enkel verfluchen merkel ampel friedrich merz ehrenwert'

In [ ]:
df_afd_2024_migration["text"].tolist()[487]

IndexError: list index out of range

'Werter Herr Präsident! Sehr geehrte Kolleginnen und Kollegen! Liebe Zuschauer! Als wir in der AfD-Fraktion erfahren haben, dass NOOTS im Plenum besprochen wird, war die ganze Fraktion elektrisiert. Wir haben aus fachfremden Arbeitskreisen Anfragen bekommen, ob wir Redeanteile abgeben können. Zwischenzeitlich mussten wir überlegen, ob wir siebenmal eine Minute oder doch lieber 14-mal 30 Sekunden dazu reden.\n\nGenau so hat es sich zugetragen oder so ähnlich, vielleicht war es auch ganz anders. Im Ergebnis habe ich jetzt sieben Minuten Redezeit zum Thema NOOTS. Und ich finde das toll!\n\nIch hoffe, Sie halten mich jetzt nicht für einen Nerd, wenn ich sage, dass das Thema NOOTS aus meiner Sicht ausgesprochen sexy ist.\n\nBevor ich das an einem Beispiel erkläre, möchte ich eine kurze Übersetzung geben: Das heißt so viel wie „Nationales Nur-einmal-erfassen-System“.\n\nIch will das am Beispiel des Kindergeldes erläutern; das ist hier ja schon angesprochen worden. Wenn man in Deutschland Kindergeld haben möchte, muss man als Allererstes eine Geburtsurkunde haben; ohne die gibt es kein Kindergeld. Eine Geburtsurkunde kriegt man idealerweise in der Außenstelle des Standesamtes in der Geburtsklinik, vorausgesetzt, das Kind kommt in einer Geburtsklinik zur Welt, und vorausgesetzt, diese Geburtsklinik hat eine Außenstelle des Standesamtes, und vorausgesetzt, diese Außenstelle hat zu der Zeit auch offen. Anderenfalls muss man zum Standesamt gehen. Das ist alles bewältigbar, ist unter Umständen aber relativ viel Arbeit. Wenn man die Geburtsurkunde hat, muss man als Nächstes zur Kindergeldstelle gehen. Die muss man ausfindig machen; das können unterschiedliche Stellen sein. Und da muss man noch mal einen ganz langen Antrag einreichen.\n\nWie wäre es denn, wenn man stattdessen im Krankenhaus einfach den Namen des Kindes und die Kontonummer fürs Kindergeld angeben würde und wenn man nach Hause kommt idealerweise schon die erste Kindergeldzahlung auf dem Konto wäre? Das klingt wie eine Utopie, ist es aber glücklicherweise nicht. Und um genau so was umzusetzen, brauchen wir NOOTS.\n\nDie Idee ist also: Wir erfassen ein Mal die Daten, und die werden intern weitergegeben; das ist ja gerade schon sehr gut erläutert worden. Die Idee ist nicht ganz neu. Als ich vor circa 30 Jahren meine ersten IT-Projekte geleitet habe, habe ich integrierte ERP-Systeme in Unternehmen des Mittelstandes eingeführt. Der große Vorteil dieser Systeme war, dass man den Auftrag nicht ausgedruckt und die Produktion ihn dann in ein anderes System eingetippt hat, sondern dass die Daten intern weitergereicht wurden. Vor 30 Jahren war das der Hit in der Privatwirtschaft. 30 Jahre später kommt es jetzt auch in der öffentlichen Verwaltung an. Das ist zwar spät, aber besser spät als nie an dieser Stelle.\n\nDas wird trotzdem kein Spaziergang, auch wenn dieses System in der Wirtschaft schon etabliert ist. Das sieht man allein schon daran, dass der NOOTS-Staatsvertrag nicht die einzige Grundlage ist, die wir brauchen, sondern es gibt noch so wohlklingende Gesetze wie das Identifikationsnummerngesetz, das Registermodernisierungsgesetz und das Onlinezugangsgesetz, die als Grundlage dafür dienen.\n\nDas zeigt schon, dass in der öffentlichen Verwaltung alles ein bisschen komplexer ist. Das kann man sicherlich noch deutlich vereinfachen. Aber man muss natürlich auch sagen: Da ist Sorgfalt absolut notwendig. – Wir reden hier über nicht weniger als über die digitale Identität der Bürger, um die es geht.\n\nSchon heute hat ein Identitätsdiebstahl dramatische Folgen, und man muss sich klarmachen: Er wird natürlich noch viel dramatischer, wenn wir ein System haben, wo alles untereinander ausgetauscht wird. Dann ist nämlich die komplette Identität weg, wenn sie geklaut wird. Dementsprechend sind zwei Dinge absolut notwendig: Es muss absolute Datensicherheit sichergestellt sein, und es muss auch ein Missbrauch des Staates verhindert werden.\n\nDer Weg dahin wird nicht einfach sein. Ich hatte schon mal die Ehre, an einem IT-Projekt teilzunehmen. Von 2016 bis 2021 war ich für den Berliner Bezirk Reinickendorf im Steuerungskreis des damaligen Digitalisierungsprojektes des rot-rot-grünen Senates vertreten. Der hatte sich vorgenommen, in sechs Jahren alle Verwaltungsvorgänge in Berlin zu digitalisieren – sehr ambitioniert. Von den 30 Mitgliedern hatte außer mir nur noch eine weitere Person Erfahrung in dem Bereich. Für alle anderen war es das erste Digitalisierungsprojekt. Entsprechend ist es auch gelaufen.\n\nIch habe dann – nur mal ein Beispiel – nach kurzer Zeit angemerkt, dass es mit dem Mitbestimmungs- und Datenschutzrecht in Berlin schwer werden wird, dieses Projekt überhaupt jemals umzusetzen. Das wurde dann weggewischt; ich hatte ja das falsche Parteibuch. Ein Jahr später hat dann die Projektleitung dem Steuerungskreis den Auftrag erteilt, zu prüfen, welche Änderungen im Mitbestimmungs- und Datenschutzrecht vorgenommen werden müssen, um den Projektvorgang zu beschleunigen.\n\nMan hatte es also ein Jahr lang liegen lassen. Umgesetzt wurde diese Maßnahme tatsächlich nie, wahrscheinlich weil irgendwann alle frustriert waren, da sie gemerkt haben, dass es nicht vorangeht.\n\nNun kann man aber aus gescheiterten Projekten sehr gut lernen, vor allem dann, wenn man nicht selber die Projekte in den Sand gesetzt hat, sondern andere; dann fällt es einem immer besonders leicht. Sieben Minuten reichen jetzt nicht aus, um über alle Fehler zu reden; aber das war einer.\n\nIch glaube, eine wichtige Sache kann man daraus mitnehmen: So ein Projekt läuft nicht in einer Legislaturperiode. Man sollte sich zwar ehrgeizige Ziele setzen, aber man muss sich klarmachen: Wir müssen schon mit fünf Legislaturperioden rechnen. Das heißt, es wird mehrere Regierungswechsel geben in dieser Zeit. Es macht also durchaus Sinn, das Projekt von Anfang an fraktionsübergreifend aufzusetzen, damit nicht mit jedem Regierungswechsel neu angefangen wird, sondern an diesem Projekt nahtlos weitergearbeitet werden kann.\n\nEs sei mir noch ein zweiter Hinweis erlaubt, den ich eingangs schon mal erwähnt habe: Ganz wesentlich ist die Akzeptanz in der Bevölkerung. Diese Akzeptanz werden wir nur kriegen, wenn wir einmal absolute Datensicherheit garantieren können. Es darf keinen Identitätsdiebstahl geben.\n\nWichtig ist außerdem: Dieses umfassende Mittel einer digitalen Identität darf nicht vom Staat missbraucht werden, indem sie zum Beispiel einfach gelöscht werden kann. Ich kann mir vorstellen, dass man dazu sogar das Grundgesetz ändern müsste, um hier einen Pflock einzuschlagen.\n\nDas Grundgesetz ist ja von den ursprünglichen Intentionen her ein Gesetz, das die Bürger vor einem übergriffigen Staat schützt. Damals, als noch der Horror des Nationalsozialismus ganz präsent war, hat man gesagt: Wir müssen die Bürger davor schützen. Dafür brauchen wir das Grundgesetz. – Heute wird es ja ganz anders benutzt.\n\nAn dieser Stelle: Ich drücke auf jeden Fall die Daumen, dass dieses Digitalisierungsprojekt erfolgreich ist. Ich denke, dass wir dafür einen fraktions- und legislaturperiodenübergreifenden Konsens benötigen. Die AfD ist gerne bereit, zu diesem beizutragen.'




















### Examining migration speech data from 1990-1993

In [ ]:
# Suchbegriffe zur Identifikation von Oppositionsreden (1990-1994)
migration_terms = [
    # 1. Kritik an der Verfassungsänderung & Grundrechte
    "Grundrecht auf Asyl",
    "Asylrecht erhalten",
    "Menschenwürde",
    "Genfer Flüchtlingskonvention",
    "Aushöhlung",
    "Abschottung",
    "Festung Europa",
    "Sündenbock",
    
    # 2. Reaktion auf rassistische Gewalt (Zentrales Oppositionsthema)
    "Fremdenfeindlichkeit",
    "Rassismus",
    "Rechtsextremismus",
    "Pogrom",
    "Hoyerswerda",
    "Rostock-Lichtenhagen",
    "Mölln",
    "Solingen",
    
    # 3. Kritik an der Sprache und Politik der Regierung
    "Abschreckungspolitik",
    "Wahlkampfgetöse",
    "Kapitulation",
    "Klima der Hetze",
    "Sündenbockpolitik",
    "Scheindebatte",
    
    # 4. Alternative Konzepte (Einwanderungsland-Debatte)
    "Einwanderungsland",
    "Einwanderungsgesetz",
    "Bleiberecht",
    "Fluchtursachen bekämpfen"
]


In [ ]:
def migration_score(text):
    text = str(text).lower()

    score = 0

    for term in migration_terms:
        score += text.count(term)

    return score


df_cdu_1990 = data[
    (data["Party"] == "CDU/CSU") &
    (data["date"].dt.year >1989)&
    (data["date"].dt.year <1994)

].copy()


df_cdu_1990["migration_score"] = (
    df_cdu_1990["text_preprocessed_lemmatized"]
    .apply(migration_score)
)

df_cdu_1990_migration = df_cdu_1990[df_cdu_1990["migration_score"] >= 3]

In [106]:
data.Party.unique()

array(['CDU/CSU', 'Cabinet', 'FDP', 'SPD', 'DP', 'fraktionslos', 'GRÜNE',
       'PDS', 'LINKE', 'AfD', 'SPDCDU/CSU', 'BSW', 'SPDSPD'], dtype=object)

In [5]:
# WICHTIG: Kleingeschrieben und in die lemmatisierte Grundform gebracht (meist Singular)
migration_terms = [
    # 1. Kritik an der Verfassungsänderung & Grundrechte
    "grundrecht", "asylrecht", "menschenwürde", "flüchtlingskonvention", 
    "aushöhlung", "abschottung", "festung", "sündenbock","asylmissbrauch","Wirtschaftsasyl",
    "Boot","Drittstaatenregelung","Kontingentflüchtlinge",
    "Asylbewerber","Asylkompromiss",
    
    # 2. Reaktion auf rassistische Gewalt
    "fremdenfeindlichkeit", "rassismus", "rechtsextremismus", "pogrom", 
    "hoyerswerda", "rostock", "lichtenhagen", "mölln", "solingen",
    
    # 3. Kritik an der Sprache und Politik
    "abschreckungspolitik", "wahlkampfgetöse", "kapitulation", "hetze", "scheindebatte",
    
    # 4. Alternative Konzepte
    "einwanderungsland", "einwanderungsgesetz", "bleiberecht", "fluchtursache"
]

def migration_score(text):
    # Sicherstellen, dass es ein String und kleingeschrieben ist
    text = str(text).lower()
    score = 0
    for term in migration_terms:
        score += text.count(term)
    return score

# Falls Sie DOCH die Opposition suchen: Ändern Sie "CDU/CSU" zu ["SPD", "DIE GRÜNEN", "PDS"]
df_reg_1990 = data[
    (data["Party"].isin(["CDU/CSU", "SPDCDU/CSU"])) & # Opposition filtern
    (data["date"].dt.year >= 1990) & # >= statt > um das Jahr 1990 mitzunehmen
    (data["date"].dt.year <= 1994)   # <= statt < um das Jahr 1994 mitzunehmen
].copy()

# Score berechnen
df_reg_1990["migration_score"] = df_reg_1990["text"].apply(migration_score)

# Filtern (Starten Sie testweise mit >= 1 oder >= 2, falls >= 3 zu streng ist)
df_reg_1990_migration = df_reg_1990[df_reg_1990["migration_score"] >= 3]


In [6]:
df_reg_1990_migration

,speech_identification_ent,date,period,session,pos_speechbeginning,Party,Role,governing_Party,text,text_length,...,text_preprocessed_lemmatized,text_length_preprocessed,text_length_lemmatized,date_year,date_quarter,similarity_expansive,similarity_restrictive,periode_index,modified_flag,migration_score
77455,Seesing (CDU/CSU):,1990-02-08,11,194,NaN,CDU/CSU,MdB,1.0,Herr Präsident! Meine Damen und Herren! In de...,863,...,dame haeufig menschenrechten verletzung mensch...,201,196,1990,1,-0.001182,0.043553,1990Q1,False,3
77704,Gerster (Mainz) (CDU/CSU):,1990-02-15,11,197,NaN,CDU/CSU,MdB,1.0,"Frau Kollegin Hämmerle, Sie sollten wie Gusta...",218,...,haemmerle gustav heinemann finger finger weise...,55,56,1990,1,0.058667,0.062734,1990Q1,False,3
78574,Gerster (Mainz) (CDU/CSU):,1990-04-26,11,207,NaN,CDU/CSU,MdB,1.0,"Erstens. Ich finde es bemerkenswert, daß Sie ...",1803,...,bemerkenswert vorsitzende innenausschusses bes...,494,472,1990,2,-0.135283,-0.127974,1990Q2,False,3
78911,Dr. Blens (CDU/CSU):,1990-05-31,11,214,NaN,CDU/CSU,MdB,1.0,Frau Präsidentin! Meine Damen und Herren! Las...,1306,...,dame datenschutzgesetz verfassungsschutzgesetz...,383,361,1990,2,0.111231,0.066392,1990Q2,False,8
79004,Fuchtel (CDU/CSU):,1990-05-31,11,214,NaN,CDU/CSU,MdB,1.0,Herr Präsident! Meine sehr verehrten Damen un...,934,...,dame heutige bescheiden hinweis nachholbedarf ...,212,209,1990,2,0.187406,0.190637,1990Q2,False,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94732,Hartmut Büttner (Schönebeck) (CDU/CSU):,1994-05-18,12,227,NaN,CDU/CSU,MdB,1.0,Frau Präsidentin! Meine verehrten Damen und H...,832,...,dame vorredner betonen auslaender deutschen ge...,243,243,1994,2,-0.148308,-0.195010,1994Q2,False,4
95001,Dr. Wolfgang Freiherr von Stetten (CDU/CSU):,1994-05-20,12,229,NaN,CDU/CSU,MdB,1.0,Herr Präsident! Meine sehr verehrten Damen un...,802,...,dame weitsichtig brennen asylrecht asylbewerbe...,221,216,1994,2,0.063615,0.056983,1994Q2,False,4
95136,Meinrad Belle (CDU/CSU):,1994-05-26,12,230,NaN,CDU/CSU,MdB,1.0,Herr Präsident! Meine Damen! Meine Herren! Ma...,449,...,dame zeitlang narr narr narr aussage abraham l...,107,107,1994,2,0.153301,0.144546,1994Q2,False,3
95446,Dr. Jürgen Rüttgers (CDU/CSU):,1994-06-23,12,235,NaN,CDU/CSU,MdB,1.0,Frau Präsidentin! Meine sehr verehrten Damen ...,279,...,dame fuchs bestaetigen kampf rechtsextremismus...,56,52,1994,2,0.184745,0.151646,1994Q2,False,3


In [86]:
df_reg_1990_migration["text_preprocessed_lemmatized"].tolist()[91]

'dame zeitlang narr narr narr aussage abraham lincoln unwillkuerlich unterlage nochmals jelpke gaukeln rassismus diskriminierung auslaendisch buergerin echte dame federfuehrend innenausschuss mitberatenden gruppe anwesend gaukeln tatsaechliche rechtliche gleichstellung auslaendisch buergerin barger rechtsordnung unterschiedlich staatsbuerger auslaender verzichten sachlich oeffentlich arbeitsmarkt differenzierung gaukeln festzustellenden auslaenderdiskriminierung begegnen ursache auslaenderfeindlichkeit unkenntnis angst diskriminierung tief kopf massnahme entgegenwirken bundestagsfraktion leimen effekthascherischen koalitionspartner gesetzentwurfes ernsthaft unterschiedlich deutschen lebend auslaender beibehalten integration positive veraenderung fortschritt moegen unterschiedlich groesse tempo bestreiten auslaenderrecht staatsangehoerigkeitsrecht demnaechst verabschiedend beanstanden offensive gewalt fremdenfeindlichkeit interessieren gruppierung gewaltpraevention jugendliche aufklaeru

'dame zeitlang narr narr narr aussage abraham lincoln unwillkuerlich unterlage nochmals jelpke gaukeln rassismus diskriminierung auslaendisch buergerin echte dame federfuehrend innenausschuss mitberatenden gruppe anwesend gaukeln tatsaechliche rechtliche gleichstellung auslaendisch buergerin barger rechtsordnung unterschiedlich staatsbuerger auslaender verzichten sachlich oeffentlich arbeitsmarkt differenzierung gaukeln festzustellenden auslaenderdiskriminierung begegnen ursache auslaenderfeindlichkeit unkenntnis angst diskriminierung tief kopf massnahme entgegenwirken bundestagsfraktion leimen effekthascherischen koalitionspartner gesetzentwurfes ernsthaft unterschiedlich deutschen lebend auslaender beibehalten integration positive veraenderung fortschritt moegen unterschiedlich groesse tempo bestreiten auslaenderrecht staatsangehoerigkeitsrecht demnaechst verabschiedend beanstanden offensive gewalt fremdenfeindlichkeit interessieren gruppierung gewaltpraevention jugendliche aufklaerungs integrationsmassnahmen entschieden vorgehen polizei justiz ursache bekaempfen buendelung massnahme stand offensive empfehlen beachtung abschliessend lade dame offensive gewalt fremdenfeindlichkeit mitwirken bedanken'

In [92]:
df_reg_1990_migration["text"].tolist()[93]

' Herr Kollege Singer, ich denke, daß das jetzt verabschiedete Verbrechensbekämpfungsgesetz, dem Sie zugestimmt haben, ergänzungsbedürftig ist. Wenn Sie mit der Polizei reden, dann ist ein Punkt natürlich von ausgesprochener Wesentlichkeit. Ich will das ganz vorurteilslos sagen, auch zu Herrn Kollege Dr. Hirsch: Wir müssen einmal darüber nachdenken, das zu tun, was Sie sicherlich gesagt haben und was wir auf dem Parteitag der CDU auch beschlossen haben, nämlich an die Umkehr der Beweislast heranzugehen.\nWenn wir das gemeinsam tun, dann wird dieses\nGesetz sicherlich noch effektiver sein. Ich wiederhole:\nWir haben Gesetze beschlossen, mit denen wir den\nKampf gegen die Gangster angesagt haben. Das wollen wir auch weiterhin tun.\nMeine Damen und Herren, ich darf die Liste der erfolgreichen Vorhaben fortsetzen: Wir haben das Ausländerrecht neu geordnet. Wir haben das Asylrecht novelliert. Zum Asylrecht: Unser Asylrecht ist weiterhin das großzügigste in der Welt, nur dem Mißbrauch wird e

' Herr Kollege Singer, ich denke, daß das jetzt verabschiedete Verbrechensbekämpfungsgesetz, dem Sie zugestimmt haben, ergänzungsbedürftig ist. Wenn Sie mit der Polizei reden, dann ist ein Punkt natürlich von ausgesprochener Wesentlichkeit. Ich will das ganz vorurteilslos sagen, auch zu Herrn Kollege Dr. Hirsch: Wir müssen einmal darüber nachdenken, das zu tun, was Sie sicherlich gesagt haben und was wir auf dem Parteitag der CDU auch beschlossen haben, nämlich an die Umkehr der Beweislast heranzugehen.\nWenn wir das gemeinsam tun, dann wird dieses\nGesetz sicherlich noch effektiver sein. Ich wiederhole:\nWir haben Gesetze beschlossen, mit denen wir den\nKampf gegen die Gangster angesagt haben. Das wollen wir auch weiterhin tun.\nMeine Damen und Herren, ich darf die Liste der erfolgreichen Vorhaben fortsetzen: Wir haben das Ausländerrecht neu geordnet. Wir haben das Asylrecht novelliert. Zum Asylrecht: Unser Asylrecht ist weiterhin das großzügigste in der Welt, nur dem Mißbrauch wird ein Riegel vorgeschoben, und wirklich politisch Verfolgten ist Schutz sicher. Aber, Herr Kollege Körper, ich weiß doch, wie schwer es gerade Ihrer Fraktion gefallen ist, diesem Gesetz letzten Endes mit einer Handvoll Stimmen zuzustimmen. Ich bedanke mich dafür. Aber Sie können doch auf Grund der Mehrheit im Bundesrat nicht sagen: Wer regiert hier denn eigentlich?\nIch meine, unser Asylrecht ist ein großzügiges Recht. Das wird auch durch die Entscheidung des Bundesinnenministers hinsichtlich der Abschiebung der Kurden bestätigt. Damit eines klar ist: Die türkischen Urteile gegen ihre eigenen kurdischen Abgeordneten sind ein schlimmes Zeichen für die Einschätzung der demokratischen Ordnung in der Türkei; sie verlangen eine neue Bewertung des Verhältnisses zu diesem NATO-Partner.\nAber ich denke, daß der Abschiebestopp bis zum 20. Januar genügend Gelegenheit gibt, darüber nachzudenken. Deswegen entbehren alle Vorwürfe an den Innenminister jeder Grundlage.\nDenn bisher — Sie wissen dies — ist kein einziger PKK-Anhänger von Deutschland an die Türkei ausgeliefert worden. Das ist auch die Linie des Innenministers. Wir wollen im Einzelfall entscheiden. Eine Einzelfallentscheidung ist letzten Endes generellen Regelungen vorzuziehen.\nMeine Damen und Herren, im Bereich der inneren Sicherheit müssen wir das Ausländer- und Asylrecht weiterentwickeln; denn der Friede, das wissen wir, beginnt im eigenen Haus. Im innenpolitisch geistigen Kampf um die Herrschaft muß die Gesinnung der Friedlosigkeit, die die Gewalt wollen würde, wenn sie nur könnte, verschwinden. Ich meine, der Etat des Finanzministers zeigt auf, daß wir im Bereich der Gewährleistung der inneren Sicherheit zufrieden sein können.\nIch denke erstens daran: Wir werden bis 1996 die Lücke im Bundesgrenzschutz geschlossen haben. Herzlichen Dank, Herr Bundesfinanzminister, herzlichen Dank, Herr Bundesinnenminister! Wir werden damit den Schleppern den Kampf angesagt haben. Das Schleppertum ist ein schlimmes Verbrechen. Wir müssen deswegen die Zahl an Grenzschutzbeamten erhöhen.\nAls zweites werden wir das Bundeskriminalamt personell und finanziell besser ausstatten. Ich meine, die Verbrechensbekämpfung beim, BKA hat sich bewährt.\nSchließlich wollen wir weiterhin die Länder bei den Bereitschaftspolizeien unterstützen. Das kostet Geld. Das wissen wir. Wir wissen auch, daß für die innere Sicherheit in erster Linie die Länder verantwortlich\nsind, in denen Sie oft das Sagen haben. Ich weiß, daß wir als Bund Gesetze zu beschließen haben, und wir werden dies tun.\nMeine Damen und Herren, wir brauchen z. B. dringend eine gesetzliche Regelung der Hauptverhandlungshaft. Warum haben Sie dem im Vermittlungsausschuß nicht zugestimmt? Sie wollen darüber nachdenken. Unser Ziel: Sofort festzunehmen, sofort zu verurteilen, sofort zum Strafantritt zu kommen. So Dinge wie in Oberhof dürfen sich nicht wiederholen. Das hat diese Koalition beschlossen.\nIch fordere Sie auf, darüber nachzudenken und es uns gleichzutun. Wir brauchen — richtig, Herr Kollege Körper — eine Novellierung des BKA-Gesetzes. Da ist eine effektive Verbrechensbekämpfung erforderlich. Ich verstehe nicht, warum die großen Länder sich dagegen sperren, dem BKA die Vorfeldbeobachtung zu übertragen. Aber dies, meine Damen und Herren, wird nicht ausreichen.\nWas ich Ihnen vorschlage, ist etwas, was wir bereits als Koalition beschlossen hatten. Ich halte es für dringend vonnöten, das G-10-Gesetz zu erweitern. Wir setzen den Bundesnachrichtendienst zum Einsatz gegen Terrorismus, gegen Drogen und gegen Handel mit spaltbarem Material ein. Warum beziehen wir nicht die individuelle Verbrechensbekämpfung ein? Es kann doch nicht sein, daß ein Gangster hier überwacht wird, weil er mit Drogen oder mit radioaktivem Material handelt; dieser Gangster hat eine zweite Wohnung in Frankreich, und er zieht dorthin und betreibt sein verbrecherisches Geschäft von dort. Dieser Gangster muß überwacht werden, meine Damen und Herren. Ich bitte Sie herzlich, das G10-Gesetz mit uns entsprechend zu ändern.\n— Ich spreche ja nicht von Ihnen, Herr Kollege Fischer. Ich spreche von Gangstern. Wissen Sie, Sie werden noch Gelegenheit haben, mit uns gemeinsam unverkrampft, ohne Überspanntheiten und ohne Übertreibungen an die Reformierung von Gesetzen zu gehen.\n'


## Constructing an pro-migration lexicon

In [94]:
data.Party.unique()

array(['CDU/CSU', 'Cabinet', 'FDP', 'SPD', 'DP', 'fraktionslos', 'GRÜNE',
       'PDS', 'LINKE', 'AfD', 'SPDCDU/CSU', 'BSW', 'SPDSPD'], dtype=object)

### 2010-2015

In [4]:
# WICHTIG: Auf Kleinschreibung achten. 
# Einige Begriffe wurden gekürzt (Wortstämme), um Pluralformen/Zusammensetzungen zu finden.
migration_terms_2010_2015 = [
    # 1. Humanitäre Aufnahme & Grundrechte (Fokus 2015-2018)
    "asylrecht", "menschenwürde", "genfer flüchtlingskonvention", "flüchtlingsschutz",
    "willkommenskultur", "humanitäre verantwortung", "aufnahmekapazität", "bleiberecht",
    "familiennachzug", "spurwechsel", "integration", "willkommen",
    "wir schaffen das", "humanitäre verpflichtung", "subsidiärer schutz",
    
    # 2. Kritik an Restriktionen / Obergrenzen
    "obergrenze", "abschottung", "festung europa", "sündenbock", "asylpaket",
    "transitzentrum", "anker-zentrum", "grenzschließung", "abschreckung",
    
    # 3. Antirassismus & Antidiskriminierung
    "fremdenfeindlichkeit", "rassismus", "rechtsextremismus", "hetze", "vorurteil",
    
    # 4. Globale Perspektive & Gesellschaftsentwurf
    "fluchtursache", "migrationspakt", "un-migrationspakt", "fluchtursachenbekämpfung",
    "einwanderungsgesellschaft", "europäische solidarität", "gelungene integration",
    "integrationsfähigkeit","Fachkräftezuwanderung","arbeitsmigration",
]

def migration_score(text):
    text = str(text).lower()
    score = 0
    for term in migration_terms_2010_2015:
        score += text.count(term)
    return score

# RELEVANTE PARTEIEN FÜR 2015-2018 FILTERN
# Hinweis: Prüfe in deinem Datensatz ('data'), wie die Parteien exakt geschrieben werden!
relevante_parteien = ["GRÜNE", "LINKE.", "SPD", "CDU/CSU"]

df_2010_2015 = data[
    (data["Party"].isin(relevante_parteien)) & 
    (data["date"].dt.year >= 2010) & 
    (data["date"].dt.year <= 2015)
].copy()

# Score berechnen
df_2010_2015["migration_score"] = df_2010_2015["text"].apply(migration_score)

# Filtern: Bei der langen Begriffskombination ist >= 3 ein guter Startwert
df_migration_filtered_2010_2015 = df_2010_2015[df_2010_2015["migration_score"] >= 3]

# Zum Lesen der Reden nach Relevanz sortieren (höchster Score zuerst)
df_migration_filtered_2010_2015 = df_migration_filtered_2010_2015.sort_values(by="migration_score", ascending=False)


In [178]:
df_migration_filtered_2010_2015

,speech_identification_ent,date,period,session,pos_speechbeginning,Party,Role,governing_Party,text,text_length,...,text_preprocessed_lemmatized,text_length_preprocessed,text_length_lemmatized,date_year,date_quarter,similarity_expansive,similarity_restrictive,periode_index,modified_flag,migration_score
188012,Michael Frieser (CDU/CSU):,2013-03-21,17,231,NaN,CDU/CSU,MdB,1.0,"\nDie Bekämpfung von Rassismus, Fremdenfeindli...",953,...,bekaempfung rassismus fremdenfeindlich keit an...,336,383,2013,1,0.064079,0.129879,2013Q1,False,35
195043,Gabriela Heinrich (SPD):,2014-10-17,18,61,NaN,SPD,MdB,1.0,\nSehr geehrter Herr Präsident! Meine Damen un...,1407,...,dame ren lieben mitglied rechte humanitaere du...,384,447,2014,4,-0.197466,-0.201216,2014Q4,False,27
179319,Michael Frieser (CDU/CSU):,2011-12-15,17,149,NaN,CDU/CSU,MdB,1.0,\nSprache ist ein Schlüssel zur erfolgreichen ...,1252,...,sprache schluessel erfolgreich inte gration in...,374,393,2011,4,-0.122582,-0.104750,2011Q4,False,25
188013,Daniela Kolbe (Leipzig) (SPD):,2013-03-21,17,231,NaN,SPD,MdB,0.0,"\nWir diskutieren heute, am Internationalen Ta...",1021,...,rassismus natio nalen aktionsplan rassismus we...,370,419,2013,1,-0.115184,-0.086597,2013Q1,False,25
188016,Monika Lazar (/DIE GRÜNEN):,2013-03-21,17,231,NaN,GRÜNE,MdB,0.0,"\nWir müssen Rassismus erkennen, beim Namen ne...",730,...,rassismus name nen nen konsequent aechten rass...,260,289,2013,1,-0.198532,-0.148883,2013Q1,False,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178440,Stephan Mayer (Altötting) (CDU/CSU):,2011-11-22,17,141,NaN,CDU/CSU,MdB,1.0,\nSehr geehrter Herr Präsident! Meine sehr ver...,1307,...,persoenlich sachlich vernuenftig etat innenmin...,329,368,2011,4,-0.080013,-0.053448,2011Q4,False,3
195126,Armin Schuster (Weil am Rhein) (CDU/CSU):,2014-11-05,18,62,NaN,CDU/CSU,MdB,1.0,\nSehr geehrter Herr Präsident! Meine sehr ver...,1092,...,dame systemversagen gross leid schuld laden mo...,263,302,2014,4,-0.285064,-0.255772,2014Q4,False,3
178467,Renate Künast (/DIE GRÜNEN):,2011-11-23,17,142,NaN,GRÜNE,MdB,0.0,\nHerr Präsident! Meine Damen und Herren! Nach...,2602,...,dame ser mer kel beschreiben klaeren froehlich...,559,639,2011,4,0.117566,0.077968,2011Q4,False,3
178602,Monika Lazar (/DIE GRÜNEN):,2011-11-24,17,143,NaN,GRÜNE,MdB,0.0,\nFrau Präsidentin! Liebe Kolleginnen und Koll...,794,...,lieben rechtsextrem mordserie wahr scheinlich ...,166,208,2011,4,-0.080249,-0.080159,2011Q4,False,3


In [87]:
df_migration_filtered_2010_2015["text_preprocessed_lemmatized"].tolist()[600]

'dame de batte medium oef fentlichkeit emotional disku tiert volkspartei stimmung buergerin faktum argument asyl fluechtling schueren rate ver sachlichung differenzierung linke linke werfen pauschalurteil abschottungspolitik mensch mensch eu asyl vorwerfen grenz schutzagentur menschenrechtswidrig verdanken grenzschutzagentur fron tex italienisch kuestenwache bedauerlich schrecklich tragoedie lampedusa nuar person seenot retten jawohl retten widerlegung tat sachenbehauptungen fortfuehren vorredner aufge griffen moegen konzentrieren ursache flucht migration massnahme bundesregie rung umgang asylbewerbern krisenherden herum jah ren ent scheiden mensch zumeist ausreise mass flucht migration armut hunger perspektivlosig keit fehlend existenzgrundlagen heimatland ursache mensch schwie rigen heimat antreten engagieren region herum stabilisieren waffe diplomatie gespraech hilfsangeboten paeischen migration fluechtling tatenlos historisch humanitaer fluechtling asylbewerber vorjahr angestie gen

'dame de batte medium oef fentlichkeit emotional disku tiert volkspartei stimmung buergerin faktum argument asyl fluechtling schueren rate ver sachlichung differenzierung linke linke werfen pauschalurteil abschottungspolitik mensch mensch eu asyl vorwerfen grenz schutzagentur menschenrechtswidrig verdanken grenzschutzagentur fron tex italienisch kuestenwache bedauerlich schrecklich tragoedie lampedusa nuar person seenot retten jawohl retten widerlegung tat sachenbehauptungen fortfuehren vorredner aufge griffen moegen konzentrieren ursache flucht migration massnahme bundesregie rung umgang asylbewerbern krisenherden herum jah ren ent scheiden mensch zumeist ausreise mass flucht migration armut hunger perspektivlosig keit fehlend existenzgrundlagen heimatland ursache mensch schwie rigen heimat antreten engagieren region herum stabilisieren waffe diplomatie gespraech hilfsangeboten paeischen migration fluechtling tatenlos historisch humanitaer fluechtling asylbewerber vorjahr angestie gen knapp viertel eu registrieren asylan traege bewaeltigen lands gesamteuropaeischen spanien asylbewerberantraege syrisch fluechtling verantwortungsbereitschaft monat fluechtling syrien aufnehmen zusage syrien hilfsschwerpunkt region million syrien humanitaer krisenbewaeltigung bau struktur verwenden technisch hilfswerk ver dienstvolle region trinkwasserversorgung fluechtlingscamps jor danien irak ursache bekaempfen mensch heimat verlassen ausgangspunkt europaeisch ebene engen partner asylsystem reformieren waehnen drittstaaten verbessert fluechtlingsschutz bekaempfung menschenhaendlern schleusern effizient grenzueberwachung gross solidaritaet eu migrationsdruck massnahme sachdien lich zuegig saetzliche neuausrichtung eu fluechtlingspolitik fehl platz asylbewerbern wahlkreisen erfahrung bringung asylbewerbern persoenlich wahlkreis syrische tschetschenische fluechtlingsfamilien be suchen kennenlernen oefter migranten hineinzuversetzen gespraech motiv asylbewerber su chen bequem syrisch fluechtling existenziell bedrohen mensch verlaesst heimat bundesamt migration fluechtling sei nen schicksal sodass ablehnungsquoten fluechtling afghanistan irak syrien niedrig person syrien abge schieben dame gruppe ursache mi gration mo tive asylsuchenden einwohner stadt gemeinde kommuni kation beteiligte asylbewerber integrieren christ betonen berechtigen su chenden willkommen mensch gewinn ansehen gesellschaftlich engagieren integrieren bundestagspraesidentin rita persoenlich schaetze fehler fluechtling leistungsfaehig tausen kilometer schrecklich strapaze wund besitzen gross mental koerperlich staerken asyl bewerbern berechtigen nahme zugang sozialsystemen be versprechung bewerber schutzgruende zuegig ausweisen schleichen migranten aufnehmen dame moegen zusammenfassen asylbewer bern differenzieren hilfsangebot berechtigen suchende fenheit menschlichkeit begegnen testament bergpredigt heissen selig gerechtigkeit verfolgen himmelreich dank'














In [86]:
df_migration_filtered_2010_2015["text"].tolist()[600]

'\nSehr geehrte Frau Präsidentin! Meine sehr geehrten\n\nDamen und Herren Kollegen! Wir führen heute eine Debatte zu einem Thema, das in den Medien und in der Öffentlichkeit in den letzten Wochen sehr emotional diskutiert wurde. Gerade als Volkspartei nehmen wir die\nStimmung der Bürgerinnen und Bürger von allen Seiten\nimmer wieder sehr ernst. Aber wir erleben auch, dass mit\nfalschen Fakten und Argumenten bestimmte Ansichten\nin der Bevölkerung zum Thema Asyl und Flüchtlinge\nbewusst geschürt werden. Daher rate ich zu einer Versachlichung und Differenzierung der Debatte.\n\n\nLeider leistet der uns vorliegende Antrag der Linken\nhierzu keinen Beitrag. Die Linken werfen wie so oft mit\nfalschen Pauschalurteilen um sich. Da ist zum Beispiel\nvon einer Abschottungspolitik der Europäischen Union\ndie Rede.\n\n\nDabei haben rund 340 000 Menschen im Jahr 2012 und\n390 000 Menschen im Jahr 2013 in der EU einen Asylantrag gestellt.\n\nWeiterhin wird vorgeworfen, die europäische Grenzschutzag




'\nSehr geehrte Frau Präsidentin! Meine sehr geehrten\n\nDamen und Herren Kollegen! Wir führen heute eine Debatte zu einem Thema, das in den Medien und in der Öffentlichkeit in den letzten Wochen sehr emotional diskutiert wurde. Gerade als Volkspartei nehmen wir die\nStimmung der Bürgerinnen und Bürger von allen Seiten\nimmer wieder sehr ernst. Aber wir erleben auch, dass mit\nfalschen Fakten und Argumenten bestimmte Ansichten\nin der Bevölkerung zum Thema Asyl und Flüchtlinge\nbewusst geschürt werden. Daher rate ich zu einer Versachlichung und Differenzierung der Debatte.\n\n\nLeider leistet der uns vorliegende Antrag der Linken\nhierzu keinen Beitrag. Die Linken werfen wie so oft mit\nfalschen Pauschalurteilen um sich. Da ist zum Beispiel\nvon einer Abschottungspolitik der Europäischen Union\ndie Rede.\n\n\nDabei haben rund 340 000 Menschen im Jahr 2012 und\n390 000 Menschen im Jahr 2013 in der EU einen Asylantrag gestellt.\n\nWeiterhin wird vorgeworfen, die europäische Grenzschutzagentur verhalte sich menschenrechtswidrig. Dabei verdanken wir gerade der Grenzschutzagentur Frontex und der italienischen Küstenwache, dass allein vom\n3. Oktober 2013, dem Tag der bedauerlichen und\nschrecklichen Tragödie von Lampedusa, bis zum 8. Januar dieses Jahres 17 000 Personen aus Seenot gerettet\nwurden; jawohl, gerettet. Die Widerlegung falscher Tatsachenbehauptungen könnte ich hier noch fortführen;\ndoch vieles wurde bereits von meinen Vorrednern aufgegriffen.\n\nIch möchte mich in meiner Rede auf drei wesentliche\nPunkte konzentrieren: erstens die Ursache für Flucht und\nMigration, zweitens die Maßnahmen der Bundesregie\n\n\nrung und der Europäischen Union sowie drittens den\nUmgang mit Asylbewerbern vor Ort in unserem Land.\n\nErstens. Wir alle wissen, dass sich die Situation in den\nKrisenherden um Europa herum in den vergangenen Jahren leider nicht verbessert hat. In diesen Ländern entscheiden sich die Menschen zumeist aus politischen\nGründen zur Ausreise nach Europa. Nicht weniger maßgebend sind wirtschaftliche und soziale Gründe für\nFlucht und Migration. Armut, Hunger, Perspektivlosigkeit und fehlende Existenzgrundlagen im Heimatland\nsind nur einige der Ursachen, die Menschen den schwierigen Weg aus ihrer Heimat antreten lassen. Deswegen\nmüssen wir dafür sorgen und uns engagieren, dass wir\ndiese Regionen um Europa herum stabilisieren – nicht\nmit Waffen, sondern mit Diplomatie, Gesprächen und\nHilfsangeboten.\n\n\nZweitens. Was wurde in Deutschland und in der Europäischen Union im Bereich Migration und Flüchtlinge\nbereits unternommen? Wir sind nicht tatenlos geblieben.\nGrundsätzlich wird Deutschland seinen historischen und\nhumanitären Verpflichtungen gegenüber Flüchtlingen\ngerecht. Die Zahl der Asylbewerber in Deutschland ist\n2013 im Vergleich zum Vorjahr um 64 Prozent angestiegen. Im Jahr 2012 hatte unser Land rund 23 Prozent, also\nknapp ein Viertel, der in der EU registrierten Asylanträge zu bewältigen. Das ist deutlich mehr als Deutschlands Anteil an der gesamteuropäischen Bevölkerung\nvon 16 Prozent. Andere Länder wie etwa Spanien haben\nnur 0,7 Prozent der Asylbewerberanträge angenommen.\n\n\nSo viel zu dem, was Deutschland bereits geleistet hat.\n\nGerade am Beispiel der syrischen Flüchtlinge wird\ndie Verantwortungsbereitschaft unseres Landes mehr als\ndeutlich. Im vergangenen Monat hat sich Deutschland\nbereit erklärt, insgesamt 10 000 Flüchtlinge allein aus\nSyrien aufzunehmen. Neben dieser Zusage für Syrien\nlegt Deutschland seinen Hilfsschwerpunkt in die Region\nselbst. So wurde seit 2012 Unterstützung in Höhe von\n432 Millionen Euro in Syrien geleistet. Diese wird für\nhumanitäre Hilfe, zur Krisenbewältigung und zum Aufbau von Strukturen im Land verwendet. Zudem leistet\ndas Technische Hilfswerk seit Juli 2012 eine sehr verdienstvolle Arbeit in der Region, insbesondere bei der\nTrinkwasserversorgung in den Flüchtlingscamps in Jordanien und Irak.\n\n\nWir wollen die Ursachen vor Ort bekämpfen, damit die\nMenschen ihre Heimat nicht verlassen müssen. Das\nmuss Ausgangspunkt unserer Arbeit sein.\n\nEbenso arbeiten wir auf europäischer Ebene sehr eng\nmit unseren Partnern daran, das gesamte europäische\nAsylsystem zu reformieren. Fünf Punkte seien hier erwähnt: eine bessere Zusammenarbeit mit Drittstaaten,\nein verbesserter Flüchtlingsschutz, die Bekämpfung von\nMenschenhändlern und Schleusern, eine effizientere\nGrenzüberwachung sowie größere Solidarität mit den\nEU-Staaten, die unter hohem Migrationsdruck stehen.\n\n\nDiese Maßnahmen sind nach meiner Ansicht sachdienlich und sollten zügig umgesetzt werden. Eine grundsätzliche Neuausrichtung der EU-Flüchtlingspolitik, wie\nsie gefordert wird, ist fehl am Platz.\n\nDamit komme ich zu meinem dritten und letzten\nPunkt: Wie gehen wir mit Asylbewerbern hier in\nDeutschland um? Viele unserer Kollegen haben in ihren\nWahlkreisen vor Ort bereits Erfahrungen mit der Unterbringung und dem Leben von Asylbewerbern gemacht.\nVor kurzem habe ich persönlich in meinem Wahlkreis\nsyrische und tschetschenische Flüchtlingsfamilien besucht und kennengelernt.\n\n\nWir tun gut daran, uns öfter in die Lage dieser Migranten\nhineinzuversetzen, mit ihnen ins Gespräch zu kommen,\nihre Motive zu verstehen. Nicht alle Asylbewerber suchen ein bequemes Leben in unserem Land. Viele, wie\ndie syrischen Flüchtlinge, sind existenziell bedroht.\n\n\nKeiner dieser Menschen verlässt seine Heimat gern. Das\nBundesamt für Migration und Flüchtlinge wird mit seinen Entscheidungen diesen Schicksalen gerecht, sodass\ndie Ablehnungsquoten für Flüchtlinge aus Ländern wie\nAfghanistan, Irak und Syrien niedrig sind. So wurden\nbereits seit 2011 keine Personen mehr nach Syrien abgeschoben.\n\nMeine Damen und Herren, wir sollten klar zwischen\nden verschiedenen Gruppen und den Ursachen der Migration unterscheiden. In jedem Fall müssen wir die Motive der Asylsuchenden den Einwohnern in den Städten\nund Gemeinden besser erklären. Denn gute Kommunikation zwischen allen Beteiligten ist sehr wichtig, um\ngemeinsam Lösungen für die Asylbewerber vor Ort zu\nfinden und sie zu integrieren. Gerade für mich als Christ\nist es wichtig, zu betonen, dass alle berechtigt Schutz Suchenden in Deutschland willkommen sind. Wir sollten\ndiese Menschen als Gewinn für unser Land ansehen.\nViele sind bereit, hier zu arbeiten, sich gesellschaftlich\nzu engagieren und sich zu integrieren.\n\nDie ehemalige Bundestagspräsidentin Rita Süssmuth,\ndie ich persönlich sehr schätze, hat hierzu einmal gesagt:\n\nWir dürfen nicht den Fehler machen, Flüchtlinge\nnicht für leistungsfähig zu halten. Wer auf Tausenden von Kilometern schreckliche Strapazen überwunden hat, besitzt große mentale und körperliche\nStärken.\n\n\n\n\nAuf der anderen Seite sollten wir aber den Asylbewerbern, die keinen berechtigten Grund für eine Aufnahme in unserem Land haben oder gar nur hierher\nkommen, um Zugang zu unseren Sozialsystemen zu bekommen, keine falschen Versprechungen machen. Diese\nBewerber, die keine Schutzgründe haben, müssen wir\nzügig wieder ausweisen. Deutschland kann schlicht\nnicht alle Migranten dieser Welt aufnehmen.\n\nMeine Damen und Herren, zum Schluss möchte ich\nzusammenfassen: Wir müssen zwischen den Asylbewerbern genau differenzieren. Unser Hilfsangebot gilt den\nberechtigt Schutz Suchenden. Ihnen sollten wir mit Offenheit, Verständnis und Menschlichkeit begegnen; denn\nschon im Neuen Testament, in der Bergpredigt, heißt es:\n\nSelig sind, die um der Gerechtigkeit willen verfolgt\nwerden; denn ihrer ist das Himmelreich.\n\nVielen Dank.\n\n\n'




In [46]:
len(df_migration_filtered_2010_2015)

757

### 2015-2018

In [24]:
# WICHTIG: Auf Kleinschreibung achten. 
# Einige Begriffe wurden gekürzt (Wortstämme), um Pluralformen/Zusammensetzungen zu finden.
migration_terms_2015_2018 = [
    # 1. Humanitäre Aufnahme & Grundrechte (Fokus 2015-2018)
    "asylrecht", "menschenwürde", "genfer flüchtlingskonvention", "flüchtlingsschutz",
    "willkommenskultur", "humanitäre verantwortung", "aufnahmekapazität", "bleiberecht",
    "familiennachzug", "spurwechsel", "integration", "willkommen",
    "wir schaffen das", "humanitäre verpflichtung", "subsidiärer schutz",
    
    # 2. Kritik an Restriktionen / Obergrenzen
    "obergrenze", "abschottung", "festung europa", "sündenbock", "asylpaket",
    "transitzentrum", "anker-zentrum", "grenzschließung", "abschreckung",
    
    # 3. Antirassismus & Antidiskriminierung
    "fremdenfeindlichkeit", "rassismus", "rechtsextremismus", "hetze", "vorurteil",
    
    # 4. Globale Perspektive & Gesellschaftsentwurf
    "fluchtursache", "migrationspakt", "un-migrationspakt", "fluchtursachenbekämpfung",
    "einwanderungsgesellschaft", "europäische solidarität", "gelungene integration",
    "integrationsfähigkeit","Fachkräftezuwanderung","arbeitsmigration",
]

def migration_score(text):
    text = str(text).lower()
    score = 0
    for term in migration_terms_2015_2018:
        score += text.count(term)
    return score

# RELEVANTE PARTEIEN FÜR 2015-2018 FILTERN
# Hinweis: Prüfe in deinem Datensatz ('data'), wie die Parteien exakt geschrieben werden!
relevante_parteien = ["GRÜNE", "LINKE.", "SPD", "CDU/CSU"]

df_2015_2018 = data[
    (data["Party"].isin(relevante_parteien)) & 
    (data["date"].dt.year >= 2015) & 
    (data["date"].dt.year <= 2018)
].copy()

# Score berechnen
df_2015_2018["migration_score"] = df_2015_2018["text"].apply(migration_score)

# Filtern: Bei der langen Begriffskombination ist >= 3 ein guter Startwert
df_migration_filtered_2015_2018 = df_2015_2018[df_2015_2018["migration_score"] >= 3]

# Zum Lesen der Reden nach Relevanz sortieren (höchster Score zuerst)
df_migration_filtered_2015_2018 = df_migration_filtered_2015_2018.sort_values(by="migration_score", ascending=False)


In [25]:
df_migration_filtered_2015_2018

,speech_identification_ent,date,period,session,pos_speechbeginning,Party,Role,governing_Party,text,text_length,...,text_preprocessed_lemmatized,text_length_preprocessed,text_length_lemmatized,date_year,date_quarter,similarity_expansive,similarity_restrictive,periode_index,modified_flag,migration_score
202865,Volker Beck (Köln) (/DIE GRÜNEN):,2016-02-25,18,158,NaN,GRÜNE,MdB,0.0,"\nEin herzliches Willkommen, Herr Minister, in...",719,...,willkommen unse rer laufen parteivorsit zender...,182,214,2016,1,-0.053933,-0.078124,2016Q1,False,23
202852,Sebastian Hartmann (SPD):,2016-02-25,18,158,NaN,SPD,MdB,1.0,\nHerr Präsident! Meine sehr geehrten Damen un...,1129,...,dame ren willy brandt einleitung nord sued be ...,250,295,2016,1,-0.040936,-0.000653,2016Q1,False,22
200099,Uwe Schummer (CDU/CSU):,2015-09-24,18,124,NaN,CDU/CSU,MdB,1.0,\nVerehrtes Präsidium! Meine Damen! Meine Herr...,1245,...,verehrtes praesidium dame werner show sem katr...,272,338,2015,3,-0.138556,-0.173825,2015Q3,False,22
202716,Dr. Eva Högl (SPD):,2016-02-19,18,156,NaN,SPD,MdB,1.0,\nSehr geehrter Herr Präsident! Liebe Kollegin...,1033,...,lieben schrift stellerin ruth klueger bemerken...,181,222,2016,1,0.064066,0.062585,2016Q1,False,20
202867,Barbara Woltmann (CDU/CSU):,2016-02-25,18,158,NaN,CDU/CSU,MdB,1.0,\nFrau Präsidentin! Meine sehr verehrten Damen...,1252,...,dame merken vorredner geset zespaket fluechtli...,258,320,2016,1,0.025744,-0.010190,2016Q1,False,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205266,Dr. Konstantin von Notz (/DIE GRÜNEN):,2016-09-06,18,185,NaN,GRÜNE,MdB,0.0,"\n\nLieber Herr Clemens Binninger, zunächst ei...",505,...,lieber clemens binninger zuhoeren explizit gef...,102,108,2016,3,-0.077742,-0.063255,2016Q3,False,3
205244,Ralph Brinkhaus (CDU/CSU):,2016-09-06,18,185,NaN,CDU/CSU,MdB,1.0,"\nVielen Dank, Herr Präsident . – Meine Damen ...",1717,...,dank dame normalerweise haushaltsausgleich erf...,328,400,2016,3,0.023820,0.035560,2016Q3,False,3
205139,Kathrin Rösel (CDU/CSU):,2016-07-07,18,183,NaN,CDU/CSU,MdB,1.0,\nSehr geehrte Frau Präsidentin! Liebe Kollegi...,956,...,lieben alltag menschenhandel gere det beschrei...,236,273,2016,3,-0.038912,-0.060921,2016Q3,False,3
204525,Dr. Karamba Diaby (SPD):,2016-06-09,18,176,NaN,SPD,MdB,1.0,\nSehr geehrte Frau Präsidentin! Liebe Jugendl...,618,...,lieben jugendliche lieben be rufsbildungsberic...,167,182,2016,2,0.091696,0.036144,2016Q2,False,3


In [167]:
df_migration_filtered_2015_2018["text_preprocessed_lemmatized"].tolist()[128]

'lieben vierte nationale bildungsbericht ple num ziemlich re jung schwerpunkt inklusion kon krete einlassen gross finanzdebatte andermal nationale bildungs berichte anknuepfen nationale bil dungsbericht schwerpunkt integration mi gration bildung schwerpunkt inklusion naechst schwerpunkt migration bildung klamm inklusion integration grundverstaendnis erkenntnis breit durchsetzen gross glueck mensch verschiedenheit bildungschancen bildungschancen inklusion integration men schen bildungschancen inklusion ha ben schwerpunkt handlungsprogramm seitens regierungs abbilden oppositionsfraktionen treiben treiben begeistert initiative assis tierte ausbildung aufnehmen inklusions integrationsgedankens assistiert ausbildung gesetzlich verankern geld unterlegt gesamtstaatlicher klusion rheinland pfalz zuerst speziell unterstuetzungspro gramm kommune auflegen million inklusion schule gross nordrhein westfalen million kommune inklusion umge geist schrittweise verschiedenheit kind jugendliche nationale

'lieben vierte nationale bildungsbericht ple num ziemlich re jung schwerpunkt inklusion kon krete einlassen gross finanzdebatte andermal nationale bildungs berichte anknuepfen nationale bil dungsbericht schwerpunkt integration mi gration bildung schwerpunkt inklusion naechst schwerpunkt migration bildung klamm inklusion integration grundverstaendnis erkenntnis breit durchsetzen gross glueck mensch verschiedenheit bildungschancen bildungschancen inklusion integration men schen bildungschancen inklusion ha ben schwerpunkt handlungsprogramm seitens regierungs abbilden oppositionsfraktionen treiben treiben begeistert initiative assis tierte ausbildung aufnehmen inklusions integrationsgedankens assistiert ausbildung gesetzlich verankern geld unterlegt gesamtstaatlicher klusion rheinland pfalz zuerst speziell unterstuetzungspro gramm kommune auflegen million inklusion schule gross nordrhein westfalen million kommune inklusion umge geist schrittweise verschiedenheit kind jugendliche nationale bildungsbericht ertragreich viermal verbal reflektieren moegen verbleibend minute national fluechtlingskindern nationale bildungsbericht erwaehnen gemeinschaftlich philosophie kommune beruehren gefluechteten mensch kind ju gendliche dramatische negative hoffentlich positi ven station geist parteifarben sortieren boehmer beauftragt bun desregierung verkuenden zugang kindertagesstaetten binden kind papier mitbringen kind papier ha ben kindertagesstaette kind schule leistung bildungs teilhabepa kets zwi schen fluechtlingskindern kind unterschied zugang tagessen kind kuemmern schule schleswig holstein million zweimal million ih rer integrationsleistungen zusaetzli che lehrerstellen bewilligen sprache schule anderswo unterrichten massnahme million beruflich bildung bemerkenswerte initiative bouffier kretschmann dreyer fluechtling ausbildung handwerks industriebetrieb anfangen garantie ha ben zwingen ausbildung zubrechen aufenthaltstitel erfordern ausbildung fortsetzen bafoeg frue warten zugang foerde rung monat billigen veraenderung monat umdenken foerderangebote gefluechteten jung men schen ablesbar ueberaus wuenschenswert philosophie nationale bildungsberichtes frueh kindlich bildung erwachsenenbildung elementare menschenrecht sprach erwerb binden aufent haltstitel ko alitionsvertrag fluechtling geduldete zugang sprache verschaffen millio nen integrationssprachkurse mensch relativ sprache erlernen erler nen sprache mensch selbstbewusstsein versiche rung altdeutsche deutschen biografie aufmachen inklusion integration migration bil dung zusammenbringen minis terin wanka kuerzlich didacta ser bildung perspektive credo gemein samen brueckenschlag inklusion integra tion festmachen bildung perspektive naechst nationale bildungsbericht zwischenergebnis aufzeigen pisa niveau kind jugendliche migrationshintergrund steigen nachdenken potenzial unabhaengig rechtlich status bildungschancen impuls werben wis sen dank'

In [176]:
df_migration_filtered_2015_2018["text"].tolist()[146]

'Herr Präsident! Meine Damen und Herren! Wir spüren es auch in dieser Debatte: In den letzten Wochen und Monaten wurde über kaum ein Thema so erbittert diskutiert wie über das Thema des Familiennachzugs. Die Diskussion war kontrovers, teilweise heftig und auch unversöhnlich. Das ist auf der einen Seite verständlich: Familie ist etwas ganz, ganz Wichtiges. Unsere Verfassung schützt Ehe und Familie, und zwar aus gutem Grund. Überall auf der Welt gilt: Kinder gehören zu ihren Eltern, genauso wie Ehefrau und Ehemann zusammengehören. Andersherum ist aber auch nachvollziehbar, dass gerade Städte und Gemeinden auf eine Steuerung von Zuzug drängen. Denn sie sind es, die sich vor Ort um Schulplätze, Kitaplätze und Wohnungen kümmern müssen.\n\nGleichwohl sage ich bei aller Wichtigkeit dieses Themas: Manchmal konnte man in den letzten Wochen schon den Eindruck gewinnen, wir hätten in unserem Land sonst keine gravierenden Probleme. Der CSU möchte ich ganz ehrlich sagen: Die Hartnäckigkeit, die Sie

'Herr Präsident! Meine Damen und Herren! Wir spüren es auch in dieser Debatte: In den letzten Wochen und Monaten wurde über kaum ein Thema so erbittert diskutiert wie über das Thema des Familiennachzugs. Die Diskussion war kontrovers, teilweise heftig und auch unversöhnlich. Das ist auf der einen Seite verständlich: Familie ist etwas ganz, ganz Wichtiges. Unsere Verfassung schützt Ehe und Familie, und zwar aus gutem Grund. Überall auf der Welt gilt: Kinder gehören zu ihren Eltern, genauso wie Ehefrau und Ehemann zusammengehören. Andersherum ist aber auch nachvollziehbar, dass gerade Städte und Gemeinden auf eine Steuerung von Zuzug drängen. Denn sie sind es, die sich vor Ort um Schulplätze, Kitaplätze und Wohnungen kümmern müssen.\n\nGleichwohl sage ich bei aller Wichtigkeit dieses Themas: Manchmal konnte man in den letzten Wochen schon den Eindruck gewinnen, wir hätten in unserem Land sonst keine gravierenden Probleme. Der CSU möchte ich ganz ehrlich sagen: Die Hartnäckigkeit, die Sie beim Thema Familiennachzug an den Tag gelegt haben, würde ich mir hin und wieder auch bei anderen Problembereichen wie gute Pflege, faire Löhne und armutsfeste Renten wünschen.\n\nDieses Land hat sicherlich noch mehr Herausforderungen zu meistern als den geordneten Zuzug von 60\xa0000\xa0Kindern und Ehefrauen.\n\nDamit kommen wir zu den Zahlen. Die meisten Experten und Studien sagen uns: Wenn wir den Familiennachzug wieder komplett zulassen, kommen etwa 60\xa0000\xa0Menschen zu uns ins Land. Das ist die eine Position. Die andere Position in dieser Debatte lautet: Wir setzen den Familiennachzug weiter komplett aus, dann kommt keiner.\n\nJetzt kommen wir zu den Planungen von Union und SPD, nämlich 1\xa0000\xa0Menschen pro Monat ab Sommer den Familiennachzug zu ermöglichen.\n\nDas bedeutet, liebe Kolleginnen und Kollegen: bis Ende dieser Legislaturperiode etwa 40\xa0000. Wenn man ganz nüchtern diese Zahlen betrachtet\xa0– null bei einer Aussetzung, 60\xa0000 bei einer kompletten Wiederaufnahme des Familiennachzuges, und die Zahl von 40\xa0000 mittendrin\xa0–, dann wird man doch nur zu dem Schluss kommen können, den auch Herr de Maizière gezogen hat: Das ist ein Kompromiss.\xa0– Die Zahl von 40\xa0000 liegt zwischen null und 60\xa0000. Das ist ein Mittelweg zwischen den Maximalforderungen.\n\nWir haben jetzt, ab Sommer, Planbarkeit für unsere Kommunen. Kein Bürgermeister muss irgendeine Turnhalle räumen, nur weil in den nächsten Jahren 40\xa0000\xa0Menschen zu uns nach Deutschland, einem Land mit über 80\xa0Millionen\xa0Einwohnern, kommen. Aber ich sage auch ganz deutlich: 40\xa0000\xa0Menschen heißt, dass die Mehrheit der Betroffenen ihre Familienangehörigen in dieser Wahlperiode wieder wird in die Arme schließen können. Und ja, dass wir Familiennachzug in dieser Größenordnung überhaupt ermöglichen\xa0– was weit über die Verabredung von Jamaika hinausgeht\xa0–, ist ein Erfolg der Sozialdemokratie.'

















### 2021-2022

In [ ]:
# WICHTIG: Auf Kleinschreibung achten. 
# Einige Begriffe wurden gekürzt (Wortstämme), um Pluralformen/Zusammensetzungen zu finden.
migration_terms_2015_2018 = [
    # 1. Humanitäre Aufnahme & Grundrechte (Fokus 2015-2018)
    "asylrecht", "menschenwürde", "genfer flüchtlingskonvention", "flüchtlingsschutz",
    "willkommenskultur", "humanitäre verantwortung", "aufnahmekapazität", "bleiberecht",
    "familiennachzug", "spurwechsel", "integration", "willkommen",
    "wir schaffen das", "humanitäre verpflichtung", "subsidiärer schutz",
    
    # 2. Kritik an Restriktionen / Obergrenzen
    "obergrenze", "abschottung", "festung europa", "sündenbock", "asylpaket",
    "transitzentrum", "anker-zentrum", "grenzschließung", "abschreckung",
    
    # 3. Antirassismus & Antidiskriminierung
    "fremdenfeindlichkeit", "rassismus", "rechtsextremismus", "hetze", "vorurteil",
    
    # 4. Globale Perspektive & Gesellschaftsentwurf
    "fluchtursache", "migrationspakt", "un-migrationspakt", "fluchtursachenbekämpfung",
    "einwanderungsgesellschaft", "europäische solidarität", "gelungene integration",
    "integrationsfähigkeit","Fachkräftezuwanderung","arbeitsmigration",
]

def migration_score(text):
    text = str(text).lower()
    score = 0
    for term in migration_terms_2015_2018:
        score += text.count(term)
    return score

# RELEVANTE PARTEIEN FÜR 2015-2018 FILTERN
# Hinweis: Prüfe in deinem Datensatz ('data'), wie die Parteien exakt geschrieben werden!
relevante_parteien = ["GRÜNE", "LINKE.", "SPD", "CDU/CSU"]

df_2015_2018 = data[
    (data["Party"].isin(relevante_parteien)) & 
    (data["date"].dt.year >= 2015) & 
    (data["date"].dt.year <= 2018)
].copy()

# Score berechnen
df_2015_2018["migration_score"] = df_2015_2018["text"].apply(migration_score)

# Filtern: Bei der langen Begriffskombination ist >= 3 ein guter Startwert
df_migration_filtered_2015_2018 = df_2015_2018[df_2015_2018["migration_score"] >= 3]

# Zum Lesen der Reden nach Relevanz sortieren (höchster Score zuerst)
df_migration_filtered_2015_2018 = df_migration_filtered_2015_2018.sort_values(by="migration_score", ascending=False)
